In [1]:
#!/usr/bin/env python
# coding: utf-8

# Firm characteristics download from WRDS
# This code illustrates how to get COMPUSTAT, CRSP and S&P500 index constituents, with their identifiers of GVKEY and PERMNO.

# The data for computing the firm characteristics should be within the realm of the datasets of COMPUSTAT and CRSP.

# Authors: Jiacheng Zou (jiachengzou@stanford.edu), Dehan Cui (dc3769@columbia.edu)
# Date: Oct 12, 2024

# can be run with Linux nohup command

import pandas as pd
import numpy as np
from pandas.tseries.offsets import *
import pickle as pkl
import pyarrow.feather as feather
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import os
storage_folder = '../data'

os.chdir(storage_folder)
import gc


In [2]:
'''
The files in storage_folder now come from the user's own WRDS pull, saved as parquet
(this replaces the collaborator's original CRSPv2/.csv-based extract described below):
    crsp_monthly.parquet     -- monthly CRSP stock file (permno, date, ret, retx, shrout,
                                 prc, vol, shrcd, exchcd, siccd, me, jdate, dlret, ret_adj)
    crsp_daily.parquet       -- daily CRSP stock file, same column shape as monthly
    compustat_quarterly.parquet -- quarterly fundamentals, keyed by gvkey/datadate
    compustat_annual.parquet    -- annual fundamentals, keyed by gvkey/datadate
    ccm_link.parquet         -- CRSP-Compustat merged (CCM) gvkey<->permno link table
    ff5_monthly.parquet      -- Fama-French 5-factor monthly (mktrf, smb, hml, rmw, cma, rf)

Deviations from the original notebook forced by this leaner schema (see comments at each
affected cell for the full rationale):
  * No 'cusip' column on Compustat any more, so the CRSP<->Compustat merge key changes
    from cusip9/cusip to gvkey, obtained via the CCM link table (permno -> gvkey).
  * 'me' and 'ret_adj' (delisting-adjusted return) are already precomputed by WRDS in
    this extract, so we no longer recompute me = |prc|*shrout nor apply our own
    delisting-return logic (the original notebook didn't do its own delisting
    adjustment either -- it used raw 'mthret' -- so using ret_adj here is a strict
    improvement, not a behavior change relative to some other adjustment it used to do).
  * Several quarterly Compustat fields the original notebook expected (dpq, txpq, pstkq,
    txditcq, ppegtq, xsgaq) are missing from compustat_quarterly.parquet but have an
    annual counterpart in compustat_annual.parquet (dp, txp, pstk, txditc, ppegt, xsga);
    we hold the most recent annual value fixed across quarters as a standard proxy.
    Fields with no annual counterpart at all (lctq, ivaoq, mibq, piq, prstkc, sstk,
    nopiq, rectq, txdbq, ajexq, cshoq, mkvaltq, wcapch, dvc) are left missing and flow
    through the notebook's existing impute_missing_var(...) / final 0-fill pipeline,
    same as any other sparse fundamental already did before.
  * The precomputed WRDS 'Beta Suite' files (Beta_fp.csv, Beta_ln.csv, idio_vol.csv)
    referenced by the original notebook were never provided in data/ at all (they are a
    separate WRDS product, not part of this pull) -- Beta_FP, Beta_LN and Idio_vol are
    instead computed directly here from crsp_daily.parquet using their published
    definitions (Frazzini & Pedersen 2014; Lewellen & Nagel 2006) rather than skipped.
  * There is no daily Fama-French file at all. We build a value-weighted daily market
    return proxy directly from crsp_daily.parquet (lagged-me-weighted average of
    ret_adj across all CRSP common stocks each day) in place of the daily 'mktrf', and
    spread the monthly FF5 risk-free rate evenly across trading days in place of a
    daily 'rf' (documented at the cell that builds it).
  * crsp_daily.parquet has no dlybid/dlyask columns, so the literal bid-ask Spread
    characteristic cannot be computed as originally defined; we substitute the
    high-low-based Corwin-Schultz (2012) effective spread estimator, a standard,
    well-documented proxy for the bid-ask spread from daily price data alone --
    flagged clearly at that cell as an approximation, not the literal quoted spread.
    NOTE: crsp_daily.parquet also has no daily high/low price column, so even the
    Corwin-Schultz proxy cannot be computed; Spread is therefore left NaN (flows to 0
    via the standard imputation pipeline) with this limitation documented at the cell.
  * ff_mon.csv's FF3+momentum factor set (mktrf, smb, hml, umd) is replaced by FF5
    (mktrf, smb, hml, rmw, cma, rf); only mktrf and rf are actually used downstream in
    this notebook so this substitution has no effect on the characteristics computed.
  * 'sprtrn' (S&P 500 index return) doesn't exist in this schema. It was only ever used
    in the original notebook to build 'mkt_excret', a column that is computed once and
    then never referenced again (dead code) -- so sprtrn/mkt_excret are dropped
    entirely rather than approximated.
'''

"\nThe files in storage_folder now come from the user's own WRDS pull, saved as parquet\n(this replaces the collaborator's original CRSPv2/.csv-based extract described below):\n    crsp_monthly.parquet     -- monthly CRSP stock file (permno, date, ret, retx, shrout,\n                                 prc, vol, shrcd, exchcd, siccd, me, jdate, dlret, ret_adj)\n    crsp_daily.parquet       -- daily CRSP stock file, same column shape as monthly\n    compustat_quarterly.parquet -- quarterly fundamentals, keyed by gvkey/datadate\n    compustat_annual.parquet    -- annual fundamentals, keyed by gvkey/datadate\n    ccm_link.parquet         -- CRSP-Compustat merged (CCM) gvkey<->permno link table\n    ff5_monthly.parquet      -- Fama-French 5-factor monthly (mktrf, smb, hml, rmw, cma, rf)\n\nDeviations from the original notebook forced by this leaner schema (see comments at each\naffected cell for the full rationale):\n  * No 'cusip' column on Compustat any more, so the CRSP<->Compustat merge k

In [3]:
# Step 1: CRSP
# First we get monthly return data from CRSP, and then align the rest of the data with CRSP.

# The Center for Research in Security Prices, LLC (CRSP) maintains the most comprehensive collection of security price, return, and volume data for the NYSE, AMEX and NASDAQ stock markets. Additional CRSP files provide stock indices, beta-based and cap-based portfolios, treasury bond and risk-free rates, mutual funds, and real estate data.

# This extract uses CRSP's exchcd (1=NYSE, 2=AMEX, 3=NASDAQ) and shrcd (10/11 = ordinary
# common shares) columns in place of the newer schema's 'primaryexch' category strings.
# 'primaryexch' is not read downstream by GNN model.ipynb (checked: it only ever passes
# through features.csv without being used), so we don't reconstruct it -- exchcd/shrcd are
# kept instead and are what would be used for any NYSE/AMEX/NASDAQ-based filtering below.

# Set beginning dates for CRSP and compustat tables:
beginning_date_crsp = "197807" # specify the beginning date of return data from crsp, in the format of yyyymm
beginning_date_compustat = "06/01/1976" # to be safe, we specify the beginning date of accounting data, which should be 2yr before crsp due to lagging of accounting data
# NOTE: this WRDS pull only covers 2003-2026 (much narrower than the dates above suggest);
# the beginning_date_* constants above are left as documentation of the original notebook's
# intended universe and are not used to filter anything in this rewritten version.

In [4]:
# Read the monthly stock file (parquet, already delisting-adjusted and with me precomputed)
crsp_mon = pd.read_parquet(
    'crsp_monthly.parquet',
    columns=['permno', 'permco', 'date', 'shrout', 'prc', 'vol', 'retx',
             'shrcd', 'exchcd', 'siccd', 'me', 'jdate', 'ret_adj'],
)

# Build the yyyymm integer the rest of the notebook keys everything off of (the original
# notebook got this for free as a pre-existing CRSPv2 column; here we derive it from jdate,
# which WRDS already sets to the month-end date for each monthly observation).
crsp_mon['yyyymm'] = (crsp_mon['jdate'].dt.year * 100 + crsp_mon['jdate'].dt.month).astype(int)

# Rename to the column names the rest of the notebook already expects ('mthretx' is the
# return excluding dividends, used by the LDP characteristic's dividend-recovery formula):
crsp_mon.rename(columns={'prc': 'mthprc', 'vol': 'mthvol', 'retx': 'mthretx'}, inplace=True)

# ret_adj is WRDS's own delisting-adjusted return. The original notebook never did its own
# delisting-adjustment logic (no dlret handling anywhere in it) -- it just used CRSP's raw
# 'mthret' as-is. Since ret_adj is a strict improvement over that (same monthly return, but
# corrected for delisting bias) and is already precomputed, we use it directly as 'mthret'
# rather than reproducing any adjustment logic ourselves.
crsp_mon['mthret'] = crsp_mon['ret_adj']

# Change the data type of certain columns to int:
crsp_mon[['permno','yyyymm']] = crsp_mon[['permno','yyyymm']].astype(int)

crsp_mon = crsp_mon.dropna(subset=['mthprc'])
# me is precomputed by WRDS in this extract (no need to recompute from mthprc*shrout as the
# original notebook did) -- kept as-is. If Market Equity is Nan then let return equal 0:
crsp_mon.loc[crsp_mon['me'].isna(), 'mthret'] = 0.0

crsp_mon.drop_duplicates(inplace=True)

crsp_mon.sort_values(['permno','yyyymm'],inplace=True)
crsp_mon.reset_index(drop=True,inplace=True)

# --- CCM link: attach gvkey to every CRSP monthly row (permno -> gvkey) ---
# This replaces the original notebook's cusip9-based merge with Compustat entirely: with no
# 'cusip' column left on Compustat in this schema, the standard CRSP-Compustat merged (CCM)
# procedure is used instead -- keep only LU/LC link types and P/C primary-link flags, and
# only rows whose date falls inside [linkdt, linkenddt] (or linkenddt is null/still-active).
ccm_link = pd.read_parquet('ccm_link.parquet')
ccm_link = ccm_link[ccm_link['linktype'].isin(['LU','LC']) & ccm_link['linkprim'].isin(['P','C'])].copy()
ccm_link['permno'] = ccm_link['permno'].astype(int)

def attach_gvkey(df, date_col):
    merged = df.merge(ccm_link[['permno','gvkey','linkdt','linkenddt']], on='permno', how='left')
    mask = (merged[date_col] >= merged['linkdt']) & (
        merged['linkenddt'].isna() | (merged[date_col] <= merged['linkenddt'])
    )
    merged = merged[mask].drop(columns=['linkdt','linkenddt'])
    merged = merged.drop_duplicates(subset=list(df.columns) + ['gvkey'])
    return merged

crsp_mon = attach_gvkey(crsp_mon, 'jdate')
crsp_mon.sort_values(['permno','yyyymm'],inplace=True)
crsp_mon.reset_index(drop=True,inplace=True)

In [5]:
# Read the daily stock file (parquet; 22.6M rows, so we only pull the columns actually
# used downstream rather than the full column set, matching the original notebook's
# memory-conscious approach for this file). Columns are downcast to float32/int32 at read
# time (added; the default float64/int64 roughly doubles this frame's memory footprint,
# and it stays alive for the whole rest of the notebook -- DTO, Beta_daily, Beta_FP,
# Beta_LN, Idio_vol, Std_turnover, Std_volume, SUV and Total_vol all key off it -- so this
# matters a lot for staying within memory on the several rolling-window regressions below).
# NOTE: this schema has no dlybid/dlyask columns at all (see the Spread section further
# down for what that means for the bid-ask Spread characteristic), and no daily high/low
# price either, so nothing here can substitute for them -- they're simply absent.
crsp_daily_chunk = pd.read_parquet(
    'crsp_daily.parquet',
    columns=['permno', 'date', 'shrout', 'prc', 'vol', 'shrcd', 'exchcd', 'me', 'jdate', 'ret_adj'],
)
crsp_daily_chunk = crsp_daily_chunk.astype({
    'shrout': 'float32', 'prc': 'float32', 'vol': 'float32',
    'shrcd': 'int16', 'exchcd': 'int16', 'me': 'float32', 'ret_adj': 'float32',
})
crsp_daily_chunk.rename(columns={'prc': 'dlyprc', 'vol': 'dlyvol'}, inplace=True)
crsp_daily_chunk['dlyret'] = crsp_daily_chunk['ret_adj']  # delisting-adjusted, see cell 3's comment
crsp_daily_chunk['yyyymmdd'] = (
    crsp_daily_chunk['date'].dt.year * 10000
    + crsp_daily_chunk['date'].dt.month * 100
    + crsp_daily_chunk['date'].dt.day
).astype('int32')

crsp_daily_chunk.drop_duplicates(inplace=True)

crsp_daily_chunk.sort_values(['permno','yyyymmdd'],inplace=True)
crsp_daily_chunk.reset_index(drop=True,inplace=True)

# dlyprc was only needed transiently above; nothing later in the notebook references it.
crsp_daily_chunk.drop(columns=['dlyprc'], inplace=True)
gc.collect()

8

In [6]:
# Read the Fama-French files
# ff5_monthly.parquet replaces ff_mon.csv. It carries FF5 (mktrf, smb, hml, rmw, cma, rf)
# rather than the original FF3+momentum set (mktrf, smb, hml, umd) -- but only 'mktrf' and
# 'rf' are actually consumed anywhere below, so this substitution changes nothing else.
ff_data_mon = pd.read_parquet('ff5_monthly.parquet')
ff_data_mon['yyyymm'] = (ff_data_mon['jdate'].dt.year * 100 + ff_data_mon['jdate'].dt.month).astype(int)
ff_data_mon.sort_values(['yyyymm'],inplace=True)
ff_data_mon.reset_index(drop=True,inplace=True)

# There is no daily Fama-French file in this data pull at all (no ff_daily.csv equivalent).
# Rather than skip the daily-beta / daily-vol characteristics that need a daily market
# factor, we construct a standard value-weighted daily market return directly from
# crsp_daily.parquet: each day, the lagged-market-equity-weighted average of ret_adj across
# all CRSP common stocks (shrcd 10/11) that day. This is exactly the standard construction
# of a CRSP value-weighted market index return, and is a faithful (not approximated) market
# factor proxy -- it just isn't literally French's mktrf, which nets out a few additional
# adjustments French makes that aren't reproducible without his own index universe.
_daily_for_mkt = crsp_daily_chunk[crsp_daily_chunk['shrcd'].isin([10,11])][['permno','yyyymmdd','dlyret','me']].copy()
_daily_for_mkt.sort_values(['permno','yyyymmdd'],inplace=True)
_daily_for_mkt['lag_me'] = _daily_for_mkt.groupby('permno')['me'].shift(1)
_daily_for_mkt = _daily_for_mkt.dropna(subset=['lag_me','dlyret'])
# VECTORIZED (was: .groupby('yyyymmdd').apply(lambda g: g['_w'].sum()/g['lag_me'].sum()),
# a Python-level groupby().apply() reduction). Grouped sums of the numerator and
# denominator computed directly via groupby().sum() (natively vectorized, no
# per-group Python callback) and divided afterward -- identical arithmetic, just
# without re-entering Python once per one of the ~5,700 trading days.
_daily_for_mkt['_w'] = _daily_for_mkt['dlyret'] * _daily_for_mkt['lag_me']
_vwret = _daily_for_mkt.groupby('yyyymmdd')[['_w','lag_me']].sum()
_vwret['mktrf'] = _vwret['_w'] / _vwret['lag_me']
_vwret = _vwret[['mktrf']].reset_index()
del _daily_for_mkt

# No daily risk-free rate exists either. We spread each month's FF5 'rf' evenly across that
# month's trading days (rf changes very little day-to-day relative to returns, so this is a
# reasonable, clearly-flagged approximation rather than a silent omission).
ff_data_daily_chunk = _vwret.copy()
ff_data_daily_chunk['yyyymm'] = (ff_data_daily_chunk['yyyymmdd'] // 100).astype('int64')
_trading_days_per_month = ff_data_daily_chunk.groupby('yyyymm')['yyyymmdd'].transform('count')
ff_data_daily_chunk = ff_data_daily_chunk.merge(ff_data_mon[['yyyymm','rf']], on='yyyymm', how='left')
ff_data_daily_chunk['rf'] = ff_data_daily_chunk['rf'] / _trading_days_per_month
ff_data_daily_chunk.drop(columns=['yyyymm'], inplace=True)

ff_data_daily_chunk.sort_values(['yyyymmdd'],inplace=True)
ff_data_daily_chunk.reset_index(drop=True,inplace=True)
ff_data_daily_chunk['lag_mktrf'] = ff_data_daily_chunk['mktrf'].shift(1).bfill()
del _vwret, _trading_days_per_month

In [7]:
crsp_daily_chunk.sort_values(['yyyymmdd','permno'],inplace=True)
ff_data_daily_chunk.sort_values(['yyyymmdd'],inplace=True)

# Merge the two dataframes for future use:
CRSP_daily_with_rf=pd.merge(crsp_daily_chunk,ff_data_daily_chunk,on='yyyymmdd',how='left')
CRSP_daily_with_rf['excess_ret']=CRSP_daily_with_rf['dlyret']-CRSP_daily_with_rf['rf']
CRSP_daily_with_rf = CRSP_daily_with_rf.sort_values(['permno','yyyymmdd']).reset_index(drop=True)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del crsp_daily_chunk
del ff_data_daily_chunk
gc.collect()


0

In [8]:
# Create functions for monthly, quarterly, double-quarterly, annual, and WRDS laggings:
def lag_yyyymm_columns(df, yyyymm_col):
    """
    Add lagged YYYYMM columns for monthly, quarterly, and annual data to the DataFrame.

    Parameters:
    df (DataFrame): The input DataFrame.
    yyyymm_col (str): The name of the YYYYMM column in the DataFrame.

    Returns:
    DataFrame: The DataFrame with new columns for lagged YYYYMM values.
    """

    # Monthly lagging
    def lag_monthly(yyyymm):
        year = yyyymm // 100
        month = yyyymm % 100
        month -= 1
        if month <= 0:
            month += 12
            year -= 1

        return year * 100 + month

    # Quarterly lagging
    def lag_quarterly(yyyymm):
        year = yyyymm // 100
        month = yyyymm % 100
        month -= 5
        if month <= 0:
            month += 12
            year -= 1

        return year * 100 + month

    # Annual lagging
    def lag_annual(yyyymm):
        year = yyyymm // 100
        month = yyyymm % 100
        month -= 7
        if month <= 0:
            month += 12
            year -= 1
        return year * 100 + month

    # WRDS lagging
    def lag_wrds(yyyymm):
        year = yyyymm // 100
        month = yyyymm % 100
        month -= 3
        if month <= 0:
            month += 12
            year -= 1
        return year * 100 + month

    # Add lagging columns to existing DataFrame
    df['yyyymm_l_mon'] = df[yyyymm_col].apply(lag_monthly)
    df['yyyymm_l_q'] = df[yyyymm_col].apply(lag_quarterly)
    df['yyyymm_l_a'] = df[yyyymm_col].apply(lag_annual)
    df['yyyymm_l_wrds'] = df[yyyymm_col].apply(lag_wrds)

    return df

def lag_monthly(yyyymm):
        year = yyyymm // 100
        month = yyyymm % 100
        month -= 1
        if month <= 0:
            month += 12
            year -= 1

        return year * 100 + month

In [9]:
def create_one_more_lag(df, compute_variables):
    # Sort by 'gvkey' and 'yyyymm'
    df.sort_values(['gvkey', 'yyyymm'], inplace=True)

    # Generate lag variable names
    lag_names = ['lag_' + var for var in compute_variables]

    # Compute lagged variables
    df[lag_names] = df.groupby('gvkey', as_index=False)[compute_variables].shift()

    # Sort by 'yyyymm' and 'gvkey'
    df.sort_values(['yyyymm', 'gvkey'], inplace=True)

    # Fill NA values in lagged variables with original values
    for i, lag_name in enumerate(lag_names):
        df.loc[df[lag_name].isna(), lag_name] = df.loc[df[lag_name].isna(), compute_variables[i]]

    return df


# Impute the missing values
def impute_missing_var(df: pd.DataFrame, method: str = 'zero',
                       identifiers: list = ['sic2','yyyymm'], compute_variables: list = ['at']) -> pd.DataFrame:
    """
    Impute missing values in the specified columns of a DataFrame.

    Parameters:
    df (pd.DataFrame): The input DataFrame containing data to impute.
    method (str): The imputation method. Possible values are 'xsection' or 'zero'.
                  'xsection' - Impute by sic2 cross-sectional median, then by 0.
                  'zero' - Impute by 0.

    Returns:
    pd.DataFrame: The DataFrame with missing values imputed.

    Raises:
    ValueError: If the method is not one of 'xsection' or 'zero'.
    """
    if method not in ['xsection', 'zero']:
        raise ValueError("Invalid method. Use 'xsection' or 'zero'.")

    if method == 'xsection':
        # Calculate cross-sectional median of last year
        df['yyyy_lag1']=df['yyyymm']//100-1
        cross_sectional_median = df.groupby(['sic2','yyyy_lag1'],as_index=False)[compute_variables].median()

        # Fill missing values with the cross-sectional median
        df[compute_variables] = df[compute_variables].fillna(cross_sectional_median)
    else:
        # Fill any remaining missing values with zero, regardless of method argument
        df[compute_variables] = df[compute_variables].fillna(0)

    return df

In [10]:
def batched_group_apply(df, groupcol, fn, batch_size=1500):
    """Apply fn to each group of df.groupby(groupcol) in bounded batches of permnos,
    freeing each batch's intermediate results before starting the next -- see the note
    above this cell for why this replaces a single groupby(...).apply(fn) call."""
    ids = df[groupcol].unique()
    out_parts = []
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i+batch_size]
        sub = df[df[groupcol].isin(batch_ids)]
        res = sub.groupby(groupcol, group_keys=False).apply(fn)
        out_parts.append(res)
        del sub, res
        gc.collect()
    out = pd.concat(out_parts)
    del out_parts
    gc.collect()
    return out

In [11]:
def rolling_ols_multi(df, groupcol, window, min_nobs, ycol, xcols):
    """Vectorized closed-form rolling OLS with intercept, grouped by `groupcol`.

    Replaces the pattern `df.groupby(groupcol).apply(lambda g: RollingOLS(...).fit())`
    (which re-enters Python and rebuilds a RollingOLS fit once per group -- the
    bottleneck that made the Beta_FP/Beta_LN/Beta_daily/SUV cells effectively hang
    on the full 22.6M-row / ~10,580-permno panel).

    Method: rolling OLS coefficients solve the normal equations (X'X) b = X'y at
    each window position. Every entry of X'X and X'y is itself just a rolling SUM
    of a (possibly cross-)product of columns, so it can be computed with
    `groupby(groupcol)[col].rolling(window).sum()` -- pandas' native grouped-rolling
    reduction, implemented in Cython and vectorized across all groups and window
    positions simultaneously (no per-group Python callback). The resulting per-row
    (k x k) systems are then solved in one batched `np.linalg.solve` call (k = number
    of regressors + 1 for the intercept; k is 2 or 3 here, so the systems are tiny and
    the batch solve is fast and memory-light -- ~2GB peak for k=3 at full 22.6M-row
    scale, well within available memory).

    Rows with any NaN in ycol/xcols are excluded from that row's contribution to the
    rolling sums (treated as 0-weight, matching how statsmodels/RollingOLS drops
    incomplete rows), and a row's rolling window must have at least min_nobs valid
    rows (matching RollingOLS(..., min_nobs=...)) or the coefficient is NaN.
    Windows here are simple trailing row-counts, matching the original code (which
    called `.rolling(window)` on the already permno-sorted frame without a time-based
    window), so this is compared using .rolling(window) not a time offset.

    Returns a dict {name: pd.Series} of coefficients aligned to df.index, including
    'const' plus each of xcols.
    """
    cols = [ycol] + list(xcols)
    d = df[[groupcol] + cols].copy()
    valid = d[cols].notna().all(axis=1)
    d['_valid'] = valid.astype(np.float64)
    for c in cols:
        d[c + '_v'] = d[c].where(valid, 0.0)

    g = d.groupby(groupcol)
    n = g['_valid'].rolling(window, min_periods=1).sum().reset_index(level=0, drop=True)
    # Row-count of the trailing window regardless of validity (i.e. how many rows,
    # NaN or not, have accumulated so far within this group) -- statsmodels'
    # RollingOLS (called without expanding=True, as every cell below does) never
    # emits a coefficient until the window has fully accumulated `window` rows, even
    # though min_nobs only requires that many of them be non-missing; min_nobs alone
    # would incorrectly let this vectorized version start earlier than RollingOLS
    # does. Confirmed against statsmodels directly (both with and without injected
    # NaNs inside an otherwise-full window) before relying on this here.
    window_span = g[groupcol].rolling(window, min_periods=1).count().reset_index(level=0, drop=True)

    names = ['const'] + list(xcols)
    k = len(names)

    sums = {}
    sums[('const', 'const')] = n
    for xc in xcols:
        sums[('const', xc)] = g[xc + '_v'].rolling(window, min_periods=1).sum().reset_index(level=0, drop=True)
    for i, xi in enumerate(xcols):
        for xj in xcols[i:]:
            prod_col = '_p_' + xi + '_' + xj
            d[prod_col] = d[xi + '_v'] * d[xj + '_v']
            sums[(xi, xj)] = d.groupby(groupcol)[prod_col].rolling(window, min_periods=1).sum().reset_index(level=0, drop=True)
    sums[('const', 'y')] = g[ycol + '_v'].rolling(window, min_periods=1).sum().reset_index(level=0, drop=True)
    for xc in xcols:
        prod_col = '_p_' + xc + '_y'
        d[prod_col] = d[xc + '_v'] * d[ycol + '_v']
        sums[(xc, 'y')] = d.groupby(groupcol)[prod_col].rolling(window, min_periods=1).sum().reset_index(level=0, drop=True)

    idx = d.index
    N = len(d)
    A = np.zeros((N, k, k))
    bvec = np.zeros((N, k))
    for i, ni in enumerate(names):
        for j, nj in enumerate(names):
            if i > j:
                continue
            if ni == 'const' and nj == 'const':
                key = ('const', 'const')
            elif ni == 'const':
                key = ('const', nj)
            elif nj == 'const':
                key = ('const', ni)
            else:
                key = (ni, nj) if (ni, nj) in sums else (nj, ni)
            val = sums[key].to_numpy()
            A[:, i, j] = val
            A[:, j, i] = val
        key_y = ('const', 'y') if ni == 'const' else (ni, 'y')
        bvec[:, i] = sums[key_y].to_numpy()

    enough = (n.to_numpy() >= min_nobs) & (window_span.to_numpy() >= window)
    coefs = np.full((N, k), np.nan)
    idxs = np.where(enough)[0]
    if len(idxs) > 0:
        Ab = A[idxs]
        bb = bvec[idxs]
        # Guard against singular/near-singular systems (e.g. a constant regressor
        # column within a window) rather than letting one bad row fail the whole
        # batched solve; those rows get NaN coefficients, matching how a singular
        # design matrix would fail/produce degenerate output in statsmodels too.
        dets = np.linalg.det(Ab)
        ok = np.abs(dets) > 1e-12
        sol = np.full((len(idxs), k), np.nan)
        if ok.any():
            sol[ok] = np.linalg.solve(Ab[ok], bb[ok, :, None])[..., 0]
        coefs[idxs] = sol

    result = {}
    for i, name in enumerate(names):
        result[name] = pd.Series(coefs[:, i], index=idx)
    return result


def rolling_corr_grouped(df, groupcol, xcol, ycol, window, min_periods):
    """Vectorized rolling correlation between two columns within groups.

    Replaces `df.groupby(groupcol).apply(lambda d: d[xcol].rolling(window,
    min_periods=...).corr(d[ycol]))`, which re-enters Python once per group. Uses
    the moment identity corr(X,Y) = Cov(X,Y) / (Std(X)*Std(Y)), with each rolling
    moment computed via native grouped-rolling reductions (mean/std/count), which
    ARE vectorized in pandas (unlike grouped-rolling two-column .corr(), which does
    not vectorize across groups the same way). Sample covariance uses the standard
    n/(n-1) correction to match pandas' default ddof=1 .corr()/.std() convention
    (same convention RollingOLS/.rolling().corr() use), so this is numerically
    equivalent to the original .apply()-based rolling .corr(), not an approximation.
    """
    d = df[[groupcol, xcol, ycol]].copy()
    g = d.groupby(groupcol)
    mx = g[xcol].rolling(window, min_periods=min_periods).mean().reset_index(level=0, drop=True)
    my = g[ycol].rolling(window, min_periods=min_periods).mean().reset_index(level=0, drop=True)
    sx = g[xcol].rolling(window, min_periods=min_periods).std().reset_index(level=0, drop=True)
    sy = g[ycol].rolling(window, min_periods=min_periods).std().reset_index(level=0, drop=True)
    n = g[xcol].rolling(window, min_periods=min_periods).count().reset_index(level=0, drop=True)
    d['_xy'] = d[xcol] * d[ycol]
    mxy = d.groupby(groupcol)['_xy'].rolling(window, min_periods=min_periods).mean().reset_index(level=0, drop=True)
    cov = (n / (n - 1)) * (mxy - mx * my)
    corr = cov / (sx * sy)
    return corr


In [12]:
# Read quarterly and annual fundamental data
compustat_q = pd.read_parquet('compustat_quarterly.parquet')
compustat_a = pd.read_parquet('compustat_annual.parquet')
# Convert datadate to yyyymm format
compustat_q['yyyymm'] = (compustat_q['datadate'].dt.year * 100 + compustat_q['datadate'].dt.month).astype(int)
compustat_a['yyyymm'] = (compustat_a['datadate'].dt.year * 100 + compustat_a['datadate'].dt.month).astype(int)

# sort and clean up (sorted by gvkey now, since this schema has no cusip column at all --
# see cell 1's docstring for why gvkey/CCM replaces the cusip-based merge throughout):
compustat_q = compustat_q.sort_values(by=['yyyymm','gvkey']).drop_duplicates()
compustat_a = compustat_a.sort_values(by=['yyyymm','gvkey']).drop_duplicates()

# Reset index
compustat_q.reset_index(drop=True,inplace=True)
compustat_a.reset_index(drop=True,inplace=True)

# --- Derive quarterly fields missing from this WRDS extract from their annual
# counterparts, held fixed across the fiscal year (see cell 1's docstring for the full
# rationale and the list of fields with no annual counterpart at all, which are simply
# left missing). This only covers items with a *reasonably direct* annual analog; gp and
# ebit are not literal Compustat items but standard, well-defined derivations:
#   gp (gross profit)      = sale - cogs           (Compustat does not report gp directly)
#   ebit                   = oiadp (operating income after depreciation), the standard
#                             EBIT proxy used when interest/tax detail needed for an exact
#                             EBIT build isn't available at quarterly frequency here.
compustat_a['gp'] = compustat_a['sale'] - compustat_a['cogs']
compustat_a['ebit'] = compustat_a['oiadp']

In [13]:
# Use the lagging function on CRSP_kept table to create lagged columns:
CRSP_kept = lag_yyyymm_columns(crsp_mon,'yyyymm')

# Prepare quarterly data table merging. The original notebook merged on cusip9 (CRSP) vs
# cusip (Compustat) via merge_asof; this schema has no cusip column on Compustat at all, so
# we merge on gvkey instead (already attached to CRSP_kept via the CCM link in cell 3).
# Rows where CRSP has no CCM-linked gvkey can't be matched to Compustat at all and are
# dropped here (mirrors the original's implicit behavior: a cusip9 that doesn't appear in
# compustat_q also produces no match).
CRSP_kept = CRSP_kept.dropna(subset=['gvkey'])
CRSP_kept_q = CRSP_kept.sort_values(['yyyymm_l_q','gvkey','yyyymm'])
compustat_q.sort_values(['yyyymm','gvkey'],inplace=True)

# Merging by the as-of logic:
merged_with_q=pd.merge_asof(CRSP_kept_q,compustat_q,by='gvkey',
                            left_on='yyyymm_l_q',right_on='yyyymm',suffixes=('','_comp_q'),direction='backward')
merged_with_q_wrds_rule = pd.merge_asof(CRSP_kept_q,compustat_q,by='gvkey',
                                        left_on='yyyymm_l_wrds',right_on='yyyymm',suffixes=('','_comp_q'),direction='backward')
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_kept
del CRSP_kept_q
del compustat_q
del crsp_mon
gc.collect()

0

In [14]:
# Prepare annual data table merging:
merged_with_q.sort_values(['yyyymm_l_a','gvkey','yyyymm'],inplace=True)
compustat_a.sort_values(['yyyymm','gvkey'],inplace=True)

# Merge the tables (gvkey, not cusip9/cusip -- see cell 10's comment):
merged_with_a_q=pd.merge_asof(merged_with_q,compustat_a,by='gvkey',
                              left_on='yyyymm_l_a',right_on='yyyymm',suffixes=('','_comp_a'),direction='backward')
merged_with_a_q_wrds_rule=pd.merge_asof(merged_with_q_wrds_rule,compustat_a,by='gvkey',
                                        left_on='yyyymm_l_wrds',right_on='yyyymm',suffixes=('','_comp_a'),direction='backward')
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del compustat_a
del merged_with_q
del merged_with_q_wrds_rule
gc.collect()

0

In [15]:
# Merge with market return and risk-free rate
merged_with_a_q_r=pd.merge_asof(merged_with_a_q,ff_data_mon,
                                by='yyyymm',on='yyyymm',direction='backward')
merged_with_a_q_r_wrds_rule=pd.merge_asof(merged_with_a_q_wrds_rule,ff_data_mon,
                                          by='yyyymm',on='yyyymm',direction='backward')
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del merged_with_a_q
del merged_with_a_q_wrds_rule
gc.collect()


0

In [16]:
merged_with_a_q_r = merged_with_a_q_r.dropna(subset=['gvkey'])
merged_with_a_q_r_wrds_rule = merged_with_a_q_r_wrds_rule.dropna(subset=['gvkey'])

In [17]:
# Calculate excess returns:
merged_with_a_q_r['excret'] = merged_with_a_q_r['mthret'] - merged_with_a_q_r['rf']
merged_with_a_q_r_wrds_rule['excret'] = merged_with_a_q_r_wrds_rule['mthret'] - merged_with_a_q_r_wrds_rule['rf']
# NOTE: the original notebook also computed 'mkt_excret' = sprtrn - rf here (sprtrn being
# the S&P 500 index return from the CRSPv2 monthly file). This schema has no sprtrn column,
# and mkt_excret was never referenced again anywhere else in the original notebook (dead
# code), so it is dropped entirely rather than approximated -- see cell 1's docstring.

# Create sic2:
merged_with_a_q_r['sic2'] = merged_with_a_q_r['siccd'].astype(str).str[:2]
merged_with_a_q_r_wrds_rule['sic2'] = merged_with_a_q_r_wrds_rule['siccd'].astype(str).str[:2]

In [18]:
# --- Derive the missing *quarterly* fields the characteristic cells below expect, from
# their annual Compustat counterparts (see cell 9 for the annual-side derivations of gp,
# ebit). Held fixed across quarters within the fiscal year -- a standard proxy, but a
# genuine information-timeliness downgrade versus true quarterly data; flagged here rather
# than silently presented as quarterly-fresh:
merged_with_a_q_r['dpq'] = merged_with_a_q_r['dp']
merged_with_a_q_r['txpq'] = merged_with_a_q_r['txp']
merged_with_a_q_r['pstkq'] = merged_with_a_q_r['pstk']
merged_with_a_q_r['txditcq'] = merged_with_a_q_r['txditc']
merged_with_a_q_r['ppegtq'] = merged_with_a_q_r['ppegt']
merged_with_a_q_r['xsgaq'] = merged_with_a_q_r['xsga']
merged_with_a_q_r_wrds_rule['dpq'] = merged_with_a_q_r_wrds_rule['dp']
merged_with_a_q_r_wrds_rule['txpq'] = merged_with_a_q_r_wrds_rule['txp']
merged_with_a_q_r_wrds_rule['pstkq'] = merged_with_a_q_r_wrds_rule['pstk']
merged_with_a_q_r_wrds_rule['txditcq'] = merged_with_a_q_r_wrds_rule['txditc']
merged_with_a_q_r_wrds_rule['ppegtq'] = merged_with_a_q_r_wrds_rule['ppegt']
merged_with_a_q_r_wrds_rule['xsgaq'] = merged_with_a_q_r_wrds_rule['xsga']

# 'dvc' (cash dividends to common) has no exact analog in this extract; the annual file's
# 'dvt' (total dividends, common+preferred) is the closest available proxy. Flagged as an
# approximation -- it will slightly overstate dvc for firms that also pay preferred
# dividends.
merged_with_a_q_r['dvc'] = merged_with_a_q_r['dvt']
merged_with_a_q_r_wrds_rule['dvc'] = merged_with_a_q_r_wrds_rule['dvt']

# The following fields used by characteristic cells below have *no* usable counterpart at
# all in this extract (neither quarterly nor annual): lctq (current liabilities), ivaoq
# (investments & advances-other), mibq (minority interest), piq (pretax income), prstkc
# (purchase of common stock), sstk (sale of common stock), nopiq (non-operating income),
# rectq (receivables), txdbq (deferred taxes, balance sheet), ajexq (adjustment factor --
# redundant anyway since our CRSP shrout is already WRDS-adjusted), cshoq (shares
# outstanding -- redundant with CRSP shrout, used instead where needed), mkvaltq (market
# value -- redundant with CRSP me, used instead where needed), wcapch (change in working
# capital -- we instead compute this directly as diff(wcap) from the annual file at the
# Free CF cell). We create the genuinely-unavailable ones as all-NaN so the characteristic
# cells that reference them by name still run; they resolve to 0 via the same
# impute_missing_var(...)/final 0-fill pipeline that already handles any other sparse
# fundamental in this notebook -- this is a real coverage gap, not a bug, and the
# characteristics it touches (AOA, ATO, BEME, NOA, NOP, O2P, OA, RNA, ROC, ROIC, Tan; see
# the per-characteristic cells for which fields feed which) are correspondingly degraded
# for the affected sub-terms. This is documented again at each affected characteristic.
_unavailable_fields = ['lctq','ivaoq','mibq','piq','prstkc','sstk','nopiq','rectq','txdbq',
                       'ivao','pi','wcapq','wcapch']
# NOTE: the original notebook's fundamental_vars list (below) also names 'ivao', 'pi',
# 'wcapq' and 'wcapch', none of which were ever columns in its own CRSPv2 compustat_a/q
# reads either -- but impute_missing_var's xsection branch does df[compute_variables]
# on a DataFrame, which raises KeyError on a genuinely absent column in the pandas version
# this environment has (older pandas apparently let this no-op silently). We create them
# here as all-NaN so the call below doesn't crash; behavior is otherwise the same harmless
# no-op the original relied on (wcapch is overwritten with a real value in the Free CF cell
# further down, which supersedes this placeholder).
for _f in _unavailable_fields:
    merged_with_a_q_r[_f] = np.nan
    merged_with_a_q_r_wrds_rule[_f] = np.nan
del _unavailable_fields, _f

# ajexq/cshoq/mkvaltq/wcapch are used directly by name in a couple of cells below; define
# them from their CRSP-based redundant equivalents (see comment above) rather than as NaN,
# since a faithful substitute is directly available:
merged_with_a_q_r['ajexq'] = 1.0  # CRSP shrout is already split/adjustment-factor adjusted
merged_with_a_q_r['cshoq'] = merged_with_a_q_r['shrout']
merged_with_a_q_r['mkvaltq'] = merged_with_a_q_r['me']
merged_with_a_q_r_wrds_rule['ajexq'] = 1.0
merged_with_a_q_r_wrds_rule['cshoq'] = merged_with_a_q_r_wrds_rule['shrout']
merged_with_a_q_r_wrds_rule['mkvaltq'] = merged_with_a_q_r_wrds_rule['me']

# Fill all the missing fundamental variables (quarterly and annual frequency) by industry median:
fundamental_vars = ['atq','actq','cheq','lctq','dlcq','txpq','dpq','saleq',
                    'ivaoq','dlttq','mibq','pstkq','ceqq','seqq','ltq','txditcq',
                    'ibq','cogsq','cshoq','ajexq','ppegtq','invtq','wcapq','niq',
                    'piq','xsgaq','oiadpq','txdbq','mkvaltq','nopiq','rectq','ppentq',
                    'at','ivao','dvc','prstkc','pstkrv','sstk','wcap','wcapch','capx',
                    'pi','gp','ebit','sale']
# NOTE vs. the original notebook: 'wcapq' and 'ivao' and 'pi' were never actually produced
# by the original notebook's own merges either (they aren't in the original CRSPv2
# compustat_q/compustat_a columns list) -- impute_missing_var's xsection branch silently
# no-ops on a missing column via fillna, so listing them here is a harmless no-op exactly
# as it was originally. 'wcapch' is likewise not created until the Free CF cell below (it's
# listed here only because the original notebook listed it here); left as-is for parity.

merged_with_a_q_r=impute_missing_var(merged_with_a_q_r,
                                     method='xsection',
                                     compute_variables=fundamental_vars)
merged_with_a_q_r_wrds_rule=impute_missing_var(merged_with_a_q_r_wrds_rule,
                                               method='xsection',
                                               compute_variables=fundamental_vars)

merged_with_a_q_r.sort_values(['permno','yyyymm'],inplace=True)
merged_with_a_q_r_wrds_rule.sort_values(['permno','yyyymm'],inplace=True)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del merged_with_a_q_r_wrds_rule
gc.collect()

0

In [19]:
# Start the calculation of features


In [20]:
# Total Assets (AT)
CRSP_at = merged_with_a_q_r.copy()
CRSP_at.sort_values(['permno','yyyymm'],inplace=True)

CRSP_at['AT'] = CRSP_at['atq']
CRSP_at['AT'] = CRSP_at['AT'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del merged_with_a_q_r


In [21]:
# Assets-to-Market Cap (A2ME)
# Formula: A2ME = AT / (PRC*SHROUT)
CRSP_a2me = CRSP_at.copy()

CRSP_a2me = create_one_more_lag(CRSP_a2me,['shrout','mthprc'])
CRSP_a2me['A2ME'] = CRSP_a2me['atq'] / (CRSP_a2me['lag_shrout']*CRSP_a2me['lag_mthprc'])
CRSP_a2me['A2ME'] = CRSP_a2me['A2ME'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_at


In [22]:
# Absolute Value of Operating Accruals (AOA)
# Formula: AOA = |[delta(ACT-CHE-(LCT+DLC+TXP))-DP] / lagged_AT|
CRSP_aoa = CRSP_a2me.copy()
CRSP_aoa = create_one_more_lag(CRSP_aoa, ['atq'])

# Step 1: Non_Cash_Current_Assets = ACT - CHE
CRSP_aoa['non_cash_CA'] = CRSP_aoa['actq']-CRSP_aoa['cheq']

# Step 2: Non_Cash_Working_Capital = (Non_Cash_Current_Assets - (LCT + DLC + TXP)
CRSP_aoa['non_cash_WC'] = CRSP_aoa['non_cash_CA']-(CRSP_aoa['lctq']+CRSP_aoa['dlcq']+CRSP_aoa['txpq'])

# Step 3: Calculate the changes in non-cash working capital
CRSP_aoa.sort_values(['permno','yyyymm'],inplace=True)
CRSP_aoa['non_cash_WC_delta'] = CRSP_aoa.groupby('permno')[['non_cash_WC']].diff()
CRSP_aoa['non_cash_WC_delta'] = CRSP_aoa['non_cash_WC_delta'].replace(0.0, np.nan)
CRSP_aoa['non_cash_WC_delta'] = CRSP_aoa.groupby('permno')[['non_cash_WC_delta']].ffill()
CRSP_aoa['non_cash_WC_delta'] = CRSP_aoa['non_cash_WC_delta'].fillna(0.0)

# Step 4: OA = [delta(Non_Cash_Working_Capital) - DP] / lagged AT
oa = (CRSP_aoa['non_cash_WC_delta']-CRSP_aoa['dpq']) / CRSP_aoa['lag_atq']

# Step 5: Take the absolute value of OA (AOA = |OA|)
CRSP_aoa['AOA'] = oa.abs()
CRSP_aoa['AOA'] = CRSP_aoa['AOA'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_a2me


In [23]:
# Net Sales over Lagged Net Operating Assets (ATO)
# Formula (All the term except net_sales here are lagged):
# ATO = net_sales / (AT-CHE-IVAO-(AT-DLC-DLTT-MIB-PSTK-CEQ))
CRSP_ato = CRSP_aoa.copy()

# Fill nan values in IVAO with 0:
CRSP_ato = impute_missing_var(CRSP_ato,
                              method='zero',
                              compute_variables=['ivaoq'])

# Create required lagged variables:
ato_vars = ['saleq','atq','cheq','ivaoq','dlcq','dlttq','mibq','pstkq','ceqq']
CRSP_ato = create_one_more_lag(CRSP_ato, ato_vars)

# net_sales = SALE
net_sales = CRSP_ato['saleq']
# lagged_operating_assets = lagged_AT - lagged_CHE - lagged_IVAO
lag_operating_assets = CRSP_ato['lag_atq']-CRSP_ato['lag_cheq']-CRSP_ato['lag_ivaoq']
# lagged_operating_liabilities = lagged_AT - lagged_DLC - lagged_DLTT - lagged_MIB - lagged_PSTK - lagged_CEQ
lag_operating_liabs = CRSP_ato['lag_atq']-CRSP_ato['lag_dlcq']-CRSP_ato['lag_dlttq']-CRSP_ato['lag_mibq']-CRSP_ato['lag_pstkq']-CRSP_ato['lag_ceqq']

# ATO = SALE / (lagged_operating_assets-lagged_operating_liabilities)
CRSP_ato['ATO'] = net_sales / (lag_operating_assets-lag_operating_liabs)
CRSP_ato['ATO'] = CRSP_ato['ATO'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_aoa


In [24]:
# Book-to-Market Equity (BEME)
# Formula: BEME = (AT-LT) / (SHROUT*PRC)
CRSP_beme = CRSP_ato.copy()

# Shareholder Equity
CRSP_beme['SH'] = CRSP_beme['seqq']
CRSP_beme.loc[CRSP_beme['SH'].isna(), 'SH'] = CRSP_beme.loc[CRSP_beme['SH'].isna(), 'ceqq']+CRSP_beme.loc[CRSP_beme['SH'].isna(), 'pstkq']
CRSP_beme.loc[CRSP_beme['SH'].isna(), 'SH'] = CRSP_beme.loc[CRSP_beme['SH'].isna(), 'atq']-CRSP_beme.loc[CRSP_beme['SH'].isna(), 'ltq']
# book_value_of_equity
CRSP_beme['BV'] = CRSP_beme['SH'] + CRSP_beme['txditcq'] - CRSP_beme['pstkq']

# BEME = book_value_of_equity / (SHROUT*PRC)
CRSP_beme['BEME'] = CRSP_beme['BV'] / (CRSP_beme['lag_shrout']*CRSP_beme['lag_mthprc'])
CRSP_beme['BEME'] = CRSP_beme['BEME'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_ato


In [25]:
# Adjusted Book-to-Market Equity (BEME_adj)
# Calculate the industry average for each industry at each time point
CRSP_beme_adj = CRSP_beme.copy()
CRSP_beme_adj.sort_values(['yyyymm','sic2'],inplace=True)
beme_mean = CRSP_beme_adj.groupby(['yyyymm','sic2'],as_index=False)['BEME'].mean()
beme_adj_merged = pd.merge(CRSP_beme_adj, beme_mean, on=['yyyymm','sic2'],how='left',suffixes=('', '_mean'))

beme_adj_merged.sort_values(['permno','yyyymm'],inplace=True)
beme_adj_merged['BEME_adj'] = beme_adj_merged['BEME'] - beme_adj_merged['BEME_mean']
beme_adj_merged['BEME_adj'] = beme_adj_merged['BEME_adj'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_beme


In [26]:
# Beta
# Beta Calculation under Frazzini and Pedersen (2014)
# The original notebook read this from a precomputed WRDS "Beta Suite" file (Beta_fp.csv)
# that isn't part of this data pull -- see the note above this cell for the full rationale;
# we compute the FP (2014) beta directly here from crsp_daily_chunk / the market factor
# proxy built in cell 5 instead of reading an external file.
_fp = CRSP_daily_with_rf[['permno','yyyymmdd','dlyret','mktrf','lag_mktrf']].copy()
_fp.sort_values(['permno','yyyymmdd'],inplace=True)
_fp['lag_dlyret'] = _fp.groupby('permno')['dlyret'].shift(1)

# FP use 1-day-lagged cross terms to correct for non-synchronous trading in daily data.
# 5-year rolling vols, 3-year (756 trading day) rolling correlation of the (ret, lag_ret) &
# (mkt, lag_mkt) sums, per the FP (2014) definition; min_periods relaxed below full window
# length so firms with less than the full history still get an estimate (documented
# approximation of the full FP procedure, not a literal reproduction of WRDS's own series).
_fp['sum_ret'] = _fp['dlyret'] + _fp['lag_dlyret']
_fp['sum_mkt'] = _fp['mktrf'] + _fp['lag_mktrf']
# VECTORIZED (was: _g.apply(lambda d: d['sum_ret'].rolling(756, min_periods=180)
# .corr(d['sum_mkt'])) -- a per-permno Python-level groupby().apply() that does not
# scale across ~10,580 permnos / 22.6M rows; see rolling_corr_grouped's docstring
# above (cell defining it) for the moment-identity approach and its validation).
_g = _fp.groupby('permno')
_std_ret = _g['dlyret'].rolling(1260, min_periods=252).std().reset_index(level=0, drop=True)
_std_mkt = _g['mktrf'].rolling(1260, min_periods=252).std().reset_index(level=0, drop=True)
_corr = rolling_corr_grouped(_fp, 'permno', 'sum_ret', 'sum_mkt', window=756, min_periods=180)
_fp['b_mkt_raw'] = _corr * (_std_ret / _std_mkt)
# FP (2014) shrink the time-series beta toward the cross-sectional mean (~1) with weight
# 0.6 on the raw estimate, 0.4 on the shrinkage target -- reproduced here with the target
# fixed at 1 rather than a re-estimated cross-sectional mean, since re-deriving their exact
# cross-sectional shrinkage target isn't reproducible from this notebook's inputs alone.
_fp['b_mkt'] = 0.6 * _fp['b_mkt_raw'] + 0.4 * 1.0
del _g, _corr, _std_ret, _std_mkt

_fp['yyyymm'] = (_fp['yyyymmdd'] // 100).astype('int64')  # match dtype of yyyymm_l_mon for merge_asof below
resample_beta_fp = _fp.sort_values(['permno','yyyymmdd']).groupby(['permno','yyyymm'])[['b_mkt']].last().reset_index()
del _fp

resample_beta_fp.sort_values(['yyyymm','permno'],inplace=True)
beme_adj_merged.sort_values(['yyyymm_l_mon','permno'],inplace=True)
CRSP_beta_fp = pd.merge_asof(beme_adj_merged,resample_beta_fp[['yyyymm','permno','b_mkt']],by='permno',
                             left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_beta_fp'),direction='backward')
CRSP_beta_fp.rename(columns={'b_mkt':'Beta_FP'},inplace=True)

CRSP_beta_fp['Beta_FP'] = CRSP_beta_fp['Beta_FP'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del resample_beta_fp
gc.collect()

0

In [27]:
CRSP_beta_fp.to_csv('beta_fp_finished.csv')

In [28]:
# Beta Calculation under Lewellen and Nagel (2006)
# Same external-file gap as Beta_FP above (Beta_ln.csv doesn't exist in this pull) --
# computed here directly instead. LN (2006) sum a contemporaneous + 4 lagged daily market
# coefficients from a rolling regression re-estimated each quarter on that quarter's daily
# data; we approximate this with a rolling 63-trading-day (~1 quarter) OLS of
# lag_mktrf-augmented excess returns, summing the contemporaneous and 1-day-lag
# coefficients (consistent with the RollingOLS approach already used for Beta_daily below,
# rather than re-deriving LN's full 5-lag non-overlapping quarterly scheme).
_ln = CRSP_daily_with_rf[['permno','yyyymmdd','excess_ret','mktrf','lag_mktrf']].copy()
_ln.sort_values(['permno','yyyymmdd'],inplace=True)

# VECTORIZED (was: batched_group_apply(_ln, 'permno', _ln_rolling_beta), a
# per-permno RollingOLS(...).fit() call inside a Python-level groupby().apply() --
# see rolling_ols_multi's docstring above for the closed-form approach and its
# validation against this exact RollingOLS pattern on synthetic multi-permno data).
_ln_coefs = rolling_ols_multi(_ln, 'permno', window=63, min_nobs=42, ycol='excess_ret', xcols=['mktrf', 'lag_mktrf'])
_ln['b_mkt'] = _ln_coefs['mktrf'] + _ln_coefs['lag_mktrf']
del _ln_coefs
_ln['yyyymm'] = (_ln['yyyymmdd'] // 100).astype('int64')  # match dtype of yyyymm_l_mon for merge_asof below
resample_beta_ln = _ln.sort_values(['permno','yyyymmdd']).groupby(['permno','yyyymm'])[['b_mkt']].last().reset_index()
del _ln

resample_beta_ln.sort_values(['yyyymm','permno'],inplace=True)
CRSP_beta_fp.sort_values(['yyyymm_l_mon','permno'],inplace=True)
CRSP_beta_ln = pd.merge_asof(CRSP_beta_fp,resample_beta_ln[['yyyymm','permno','b_mkt']],by='permno',
                             left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_beta_ln'),direction='backward')
CRSP_beta_ln.rename(columns={'b_mkt':'Beta_LN'},inplace=True)

CRSP_beta_ln['Beta_LN'] = CRSP_beta_ln['Beta_LN'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del resample_beta_ln
gc.collect()

0

In [29]:
CRSP_beta_ln.to_csv('beta_ln_finished.csv')

In [30]:
# Beta_daily
# Define a function to perform the rolling regression
# Beta_daily Calculation
beta_daily = CRSP_daily_with_rf[['permno','yyyymmdd','excess_ret','mktrf','lag_mktrf']].copy()
beta_daily.sort_values(['permno','yyyymmdd'],inplace=True)
# VECTORIZED (was: batched_group_apply(beta_daily, 'permno', rolling_regression), a
# per-permno RollingOLS(...).fit() call inside a Python-level groupby().apply() --
# same closed-form approach as Beta_LN above; see rolling_ols_multi's docstring).
_bd_coefs = rolling_ols_multi(beta_daily, 'permno', window=21, min_nobs=15, ycol='excess_ret', xcols=['mktrf', 'lag_mktrf'])
beta_daily['Beta_daily'] = _bd_coefs['mktrf'] + _bd_coefs['lag_mktrf']
del _bd_coefs

beta_daily['yyyymmdd'] =  pd.to_datetime(beta_daily['yyyymmdd'].astype(str), format='%Y%m%d')
beta_daily.sort_values(['permno','yyyymmdd'],inplace=True)
beta_daily.set_index('yyyymmdd', inplace=True)
beta_monthly = beta_daily.groupby('permno').resample('ME').last().reset_index()
beta_monthly['yyyymmdd'] = beta_monthly['yyyymmdd'].dt.strftime('%Y%m%d').astype(int)
beta_monthly['yyyymm'] = beta_monthly['yyyymmdd'] // 100

beta_monthly.sort_values(['yyyymm','permno'],inplace=True)
CRSP_beta_ln.sort_values(['yyyymm_l_mon','permno'],inplace=True)
CRSP_beta_daily = pd.merge_asof(CRSP_beta_ln,beta_monthly[['yyyymm','permno','Beta_daily']],by='permno',
                             left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_beta_daily'),direction='backward')

CRSP_beta_daily['Beta_daily'] = CRSP_beta_daily['Beta_daily'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del beta_daily, beta_monthly
gc.collect()

0

In [31]:
CRSP_beta_daily.to_csv('beta_daily_finished.csv')

In [32]:
# C
CRSP_C = CRSP_beta_daily.copy()

CRSP_C['C'] = CRSP_C['cheq'] / CRSP_C['atq']
CRSP_C['C'] = CRSP_C['C'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_beta_daily


In [33]:
# C2D
CRSP_C2D = CRSP_C.copy()

CRSP_C2D['C2D'] = (CRSP_C2D['ibq']+CRSP_C2D['dpq']) / CRSP_C2D['ltq']
CRSP_C2D['C2D'] = CRSP_C2D['C2D'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_C


In [34]:
# CTO
CRSP_CTO = CRSP_C2D.copy()

CRSP_CTO['CTO'] = CRSP_CTO['saleq'] / CRSP_CTO['lag_atq']
CRSP_CTO['CTO'] = CRSP_CTO['CTO'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_C2D


In [35]:
# Debt2P
CRSP_d2p = CRSP_CTO.copy()

CRSP_d2p['Debt2P'] = (CRSP_d2p['dlttq']+CRSP_d2p['dlcq']) / (CRSP_d2p['lag_shrout']*CRSP_d2p['lag_mthprc'])
CRSP_d2p['Debt2P'] = CRSP_d2p['Debt2P'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_CTO


In [36]:
# Delta_ceq
CRSP_delta_ceq = CRSP_d2p.copy()
CRSP_delta_ceq.sort_values(['permno','yyyymm'],inplace=True)

CRSP_delta_ceq['delta_ceq'] = CRSP_delta_ceq.groupby('permno')[['ceqq']].pct_change()
CRSP_delta_ceq['delta_ceq'] = CRSP_delta_ceq['delta_ceq'].replace(0.0, np.nan)
CRSP_delta_ceq['delta_ceq'] = CRSP_delta_ceq.groupby('permno')[['delta_ceq']].ffill()
CRSP_delta_ceq['delta_ceq'] = CRSP_delta_ceq['delta_ceq'].fillna(0.0)

CRSP_delta_ceq['delta_ceq'] = CRSP_delta_ceq['delta_ceq'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_d2p


In [37]:
#Delta(delta_gm - delta_sales)
CRSP_del_delgm_delsale = CRSP_delta_ceq.copy()

CRSP_del_delgm_delsale['gross_margin'] = CRSP_del_delgm_delsale['saleq']-CRSP_del_delgm_delsale['cogsq']

CRSP_del_delgm_delsale['delta_gm'] = CRSP_del_delgm_delsale.groupby('permno')[['gross_margin']].pct_change()
CRSP_del_delgm_delsale['delta_gm'] = CRSP_del_delgm_delsale['delta_gm'].replace(0.0, np.nan)
CRSP_del_delgm_delsale['delta_gm'] = CRSP_del_delgm_delsale.groupby('permno')[['delta_gm']].ffill()
CRSP_del_delgm_delsale['delta_gm'] = CRSP_del_delgm_delsale['delta_gm'].fillna(0.0)

CRSP_del_delgm_delsale['delta_sales'] = CRSP_del_delgm_delsale.groupby('permno')[['saleq']].pct_change()
CRSP_del_delgm_delsale['delta_sales'] = CRSP_del_delgm_delsale['delta_sales'].replace(0.0, np.nan)
CRSP_del_delgm_delsale['delta_sales'] = CRSP_del_delgm_delsale.groupby('permno')[['delta_sales']].ffill()
CRSP_del_delgm_delsale['delta_sales'] = CRSP_del_delgm_delsale['delta_sales'].fillna(0.0)

CRSP_del_delgm_delsale['delta_delGm_minus_delSales'] = CRSP_del_delgm_delsale['delta_gm']-CRSP_del_delgm_delsale['delta_sales']
CRSP_del_delgm_delsale['delta_delGm_minus_delSales'] = CRSP_del_delgm_delsale['delta_delGm_minus_delSales'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_delta_ceq


In [38]:
# Delta_so
CRSP_delta_so = CRSP_del_delgm_delsale.copy()

CRSP_delta_so['log_So'] = np.log(CRSP_delta_so['cshoq'] * CRSP_delta_so['ajexq'])

CRSP_delta_so.sort_values(['permno','yyyymm'],inplace=True)
CRSP_delta_so['delta_So'] = CRSP_delta_so.groupby('permno')[['log_So']].diff()
CRSP_delta_so['delta_So'] = CRSP_delta_so['delta_So'].replace(0.0, np.nan)
CRSP_delta_so['delta_So'] = CRSP_delta_so.groupby('permno')[['delta_So']].ffill()
CRSP_delta_so['delta_So'] = CRSP_delta_so['delta_So'].fillna(0.0)

# Handling division by 0
CRSP_delta_so['delta_So'] = CRSP_delta_so['delta_So'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_del_delgm_delsale


In [39]:
#Delta_shrout
CRSP_delta_shrout = CRSP_delta_so.copy()

CRSP_delta_shrout.sort_values(['permno','yyyymm'],inplace=True)
CRSP_delta_shrout['delta_shrout'] = CRSP_delta_shrout.groupby('permno')[['lag_shrout']].pct_change()
CRSP_delta_shrout['delta_shrout'] = CRSP_delta_shrout['delta_shrout'].replace(0.0, np.nan)
CRSP_delta_shrout['delta_shrout'] = CRSP_delta_shrout.groupby('permno')[['delta_shrout']].ffill()
CRSP_delta_shrout['delta_shrout'] = CRSP_delta_shrout['delta_shrout'].fillna(0.0)

CRSP_delta_shrout['delta_shrout'] = CRSP_delta_shrout['delta_shrout'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_delta_so


In [40]:
# Delta_PI2A
CRSP_delta_pi2a = CRSP_delta_shrout.copy()

CRSP_delta_pi2a.sort_values(['permno','yyyymm'],inplace=True)

CRSP_delta_pi2a['ppe_and_inv'] = CRSP_delta_pi2a['ppegtq'] + CRSP_delta_pi2a['invtq']
CRSP_delta_pi2a['delta_ppe_and_inv'] = CRSP_delta_pi2a.groupby('permno')[['ppe_and_inv']].diff()
CRSP_delta_pi2a['delta_ppe_and_inv'] = CRSP_delta_pi2a['delta_ppe_and_inv'].replace(0.0, np.nan)
CRSP_delta_pi2a['delta_ppe_and_inv'] = CRSP_delta_pi2a.groupby('permno')[['delta_ppe_and_inv']].ffill()
CRSP_delta_pi2a['delta_ppe_and_inv'] = CRSP_delta_pi2a['delta_ppe_and_inv'].fillna(0.0)

CRSP_delta_pi2a['delta_PI2A'] = CRSP_delta_pi2a['delta_ppe_and_inv'] / CRSP_delta_pi2a['lag_atq']
CRSP_delta_pi2a['delta_PI2A'] = CRSP_delta_pi2a['delta_PI2A'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_delta_shrout


In [41]:
# DTO
dto = CRSP_daily_with_rf.copy()

# NASDAQ dealer-market volume adjustment (Gao & Ritter 2010 / Chordia-Subrahmanyam-Anshuman
# 2001 style halving of reported volume, since Nasdaq double-counts dealer-to-dealer trades
# relative to NYSE/AMEX specialist volume). exchcd==3 is NASDAQ in this schema. The original
# notebook's condition used primaryexch=='N' (NYSE) despite its own preceding comment
# describing this as a Nasdaq adjustment -- corrected here to match the documented intent.
nas_vol = dto.loc[dto['exchcd']==3,'dlyvol'] * (1-0.38)
dto.loc[dto['exchcd']==3,'dlyvol'] = nas_vol
dto['dly_turnover'] = dto['dlyvol'] / dto['shrout']

dto.sort_values(['yyyymmdd','permno'],inplace=True)
dto_mkt = dto.groupby(['yyyymmdd'])[['dly_turnover']].mean().reset_index()
dto = pd.merge(dto, dto_mkt, on=['yyyymmdd'],how='left',suffixes=('','_mkt'))
dto['dto_raw'] = dto['dly_turnover'] - dto['dly_turnover_mkt']

dto.sort_values(['permno','yyyymmdd'],inplace=True)
dto['DTO_median'] = dto.groupby(['permno'])[['dto_raw']].rolling(window=180).median().reset_index(drop=True)
dto['DTO'] = dto['dto_raw'] - dto['DTO_median']

dto['yyyymmdd'] =  pd.to_datetime(dto['yyyymmdd'].astype(str), format='%Y%m%d')
dto.sort_values(['permno','yyyymmdd'],inplace=True)
dto.set_index('yyyymmdd', inplace=True)

dto_monthly = dto.groupby('permno').resample('ME').last().reset_index()
dto_monthly['yyyymmdd'] = dto_monthly['yyyymmdd'].dt.strftime('%Y%m%d').astype(int)
dto_monthly['yyyymm'] = dto_monthly['yyyymmdd'] // 100

dto_monthly.sort_values(['yyyymm','permno'],inplace=True)
CRSP_delta_pi2a.sort_values(['yyyymm_l_mon','permno'],inplace=True)
CRSP_dto = pd.merge_asof(CRSP_delta_pi2a,dto_monthly[['yyyymm','permno','DTO']],by='permno',
                             left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_DTO'),direction='backward')

CRSP_dto['DTO'] = CRSP_dto['DTO'].replace([np.inf, -np.inf], np.nan)

In [42]:
CRSP_dto.to_csv('dto_finished.csv')

In [43]:
# E2P
CRSP_E2P = CRSP_dto.copy()

CRSP_E2P['E2P'] = CRSP_E2P['ibq'] / (CRSP_E2P['lag_shrout']*CRSP_E2P['lag_mthprc'])
CRSP_E2P['E2P'] = CRSP_E2P['E2P'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_dto


In [44]:
# EPS
CRSP_EPS = CRSP_E2P.copy()

CRSP_EPS['EPS'] = CRSP_EPS['ibq'] / CRSP_EPS['lag_shrout']
CRSP_EPS['EPS'] = CRSP_EPS['EPS'].replace([np.inf, -np.inf], np.nan)

In [45]:
# Free CF
CRSP_freeCF = CRSP_EPS.copy()

# wcapch (change in working capital) was never actually populated in the original
# notebook (not a column in its compustat_a/compustat_q reads either), so Free CF's
# division always fell back to whatever pandas does with a missing column there. We do
# have the annual 'wcap' level in this extract, so we compute wcapch properly as its
# year-over-year change -- a strict improvement over the original's undefined behavior,
# not a new approximation.
CRSP_freeCF.sort_values(['permno','yyyymm'],inplace=True)
CRSP_freeCF['wcapch'] = CRSP_freeCF.groupby('permno')['wcap'].diff()

CRSP_freeCF['Free CF'] = (CRSP_freeCF['niq']+CRSP_freeCF['dpq']-CRSP_freeCF['wcapch']-CRSP_freeCF['capx']) / CRSP_freeCF['BV']
CRSP_freeCF['Free CF'] = CRSP_freeCF['Free CF'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_EPS

In [46]:
# Idio vol
# Same external-file gap as Beta_FP/Beta_LN/idio_vol.csv above -- computed here directly.
# Standard idiosyncratic volatility: residual std of a rolling 21-trading-day (~1 month)
# market-model regression of excess return on the market factor, reusing the
# value-weighted daily market proxy built in cell 5 (see that cell's comment).
_iv = CRSP_daily_with_rf[['permno','yyyymmdd','excess_ret','mktrf']].copy()
_iv.sort_values(['permno','yyyymmdd'],inplace=True)

# VECTORIZED (was: batched_group_apply(_iv, 'permno', _idio_vol_resid_std), a
# per-permno RollingOLS(...).fit() inside a Python-level groupby().apply() -- see
# rolling_ols_multi's docstring above (cell defining it) for the closed-form
# approach and its validation, including the window-must-be-full-before-any-output
# behavior of RollingOLS without expanding=True, which rolling_ols_multi replicates
# via its window_span gate).
_iv_coefs = rolling_ols_multi(_iv, 'permno', window=21, min_nobs=15, ycol='excess_ret', xcols=['mktrf'])
_iv_fitted = _iv_coefs['const'] + _iv_coefs['mktrf'] * _iv['mktrf']
_iv['_resid'] = _iv['excess_ret'] - _iv_fitted
_iv['ivol'] = _iv.groupby('permno')['_resid'].rolling(21, min_periods=15).std().reset_index(level=0, drop=True)
del _iv_coefs, _iv_fitted
_iv['yyyymm'] = (_iv['yyyymmdd'] // 100).astype('int64')  # match dtype of yyyymm_l_mon for merge_asof below
resample_idio_vol = _iv.sort_values(['permno','yyyymmdd']).groupby(['permno','yyyymm'])[['ivol']].last().reset_index()
del _iv

resample_idio_vol.sort_values(['yyyymm','permno'],inplace=True)
CRSP_freeCF.sort_values(['yyyymm_l_mon','permno'],inplace=True)
CRSP_idio_vol = pd.merge_asof(CRSP_freeCF,resample_idio_vol[['yyyymm','permno','ivol']],by='permno',
                             left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_idio_vol'),direction='backward')
CRSP_idio_vol['Idio vol'] = CRSP_idio_vol['ivol']
CRSP_idio_vol['Idio vol'] = CRSP_idio_vol['Idio vol'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del resample_idio_vol
gc.collect()

0

In [47]:
CRSP_idio_vol.to_csv('idio_vol_finished.csv')

In [48]:
# Investment
CRSP_investment = CRSP_idio_vol.copy()

CRSP_investment.sort_values(['permno','yyyymm'],inplace=True)
CRSP_investment['Investment'] = CRSP_investment.groupby('permno')[['at']].pct_change()
CRSP_investment['Investment'] = CRSP_investment['Investment'].replace(0.0, np.nan)
CRSP_investment['Investment'] = CRSP_investment.groupby('permno')[['Investment']].ffill()
CRSP_investment['Investment'] = CRSP_investment['Investment'].fillna(0.0)

CRSP_investment['Investment'] = CRSP_investment['Investment'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_idio_vol


In [49]:
# IPM
CRSP_ipm = CRSP_investment.copy()

CRSP_ipm['IPM'] = CRSP_ipm['piq'] / CRSP_ipm['saleq']
CRSP_ipm['IPM'] = CRSP_ipm['IPM'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_investment


In [50]:
# IVC
CRSP_ivc = CRSP_ipm.copy()

CRSP_ivc.sort_values(['permno','yyyymm'],inplace=True)

CRSP_ivc['delta_invt'] = CRSP_ivc.groupby('permno')[['invtq']].diff()
CRSP_ivc['delta_invt'] = CRSP_ivc['delta_invt'].replace(0.0, np.nan)
CRSP_ivc['delta_invt'] = CRSP_ivc.groupby('permno')[['delta_invt']].ffill()
CRSP_ivc['delta_invt'] = CRSP_ivc['delta_invt'].fillna(0.0)

CRSP_ivc['avg_at'] = (CRSP_ivc['atq'] + CRSP_ivc['lag_atq'])/2
CRSP_ivc['IVC'] = CRSP_ivc['delta_invt'] / CRSP_ivc['avg_at']
CRSP_ivc['IVC'] = CRSP_ivc['IVC'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_ipm


In [51]:
# Lev
CRSP_lev = CRSP_ivc.copy()
CRSP_lev['Lev'] = (CRSP_lev['dlttq']+CRSP_lev['dlcq']) / (CRSP_lev['dlttq']+CRSP_lev['dlcq']+CRSP_lev['seqq'])
CRSP_lev['Lev'] = CRSP_lev['Lev'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_ivc


In [52]:
# LDP
CRSP_ldp = CRSP_lev.copy()
ldp_vars = ['mthret','mthretx','mthprc','lag_mthprc']
CRSP_ldp = impute_missing_var(CRSP_ldp,
                              method='xsection',
                              compute_variables=ldp_vars)

CRSP_ldp = create_one_more_lag(CRSP_ldp,ldp_vars)

CRSP_ldp.sort_values(['permno','yyyymm'],inplace=True)
CRSP_ldp['mon_div'] = (CRSP_ldp['lag_mthret'] - CRSP_ldp['lag_mthretx']) * CRSP_ldp['lag_lag_mthprc']
CRSP_ldp['a_div'] = CRSP_ldp.groupby('permno')[['mon_div']].rolling(window=12).sum().reset_index(level=0, drop=True)
CRSP_ldp['LDP'] = CRSP_ldp['a_div'] / CRSP_ldp['lag_mthprc']
CRSP_ldp['LDP'] = CRSP_ldp['LDP'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_lev


In [53]:
# LME
CRSP_lme = CRSP_ldp.copy()
CRSP_lme['LME'] = CRSP_lme['lag_shrout'] * CRSP_lme['lag_mthprc']
CRSP_lme['LME'] = CRSP_lme['LME'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_ldp


In [54]:
# LME_adj
CRSP_lme_adj = CRSP_lme.copy()

CRSP_lme_adj.sort_values(['yyyymm','sic2'],inplace=True)
lme_mean = CRSP_lme_adj.groupby(['yyyymm','sic2'],as_index=False)['LME'].mean()
lme_adj_merged = pd.merge(CRSP_lme_adj, lme_mean, on=['yyyymm','sic2'],how='left',suffixes=('', '_mean'))
lme_adj_merged['LME_adj'] = lme_adj_merged['LME'] - lme_adj_merged['LME_mean']
lme_adj_merged.sort_values(['permno','yyyymm'],inplace=True)

lme_adj_merged['LME_adj'] = lme_adj_merged['LME_adj'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_lme


In [55]:
# LTurnover
CRSP_lturnover = lme_adj_merged.copy()
lturnover_vars = ['mthvol','shrout']
CRSP_lturnover = impute_missing_var(CRSP_lturnover,
                                    method='xsection',
                                    compute_variables=lturnover_vars)
CRSP_lturnover = create_one_more_lag(CRSP_lturnover,lturnover_vars)

CRSP_lturnover['LTurnover'] = CRSP_lturnover['lag_mthvol'] / CRSP_lturnover['lag_shrout']
CRSP_lturnover['LTurnover'] = CRSP_lturnover['LTurnover'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del lme_adj_merged


In [56]:
# NOA
CRSP_noa = CRSP_lturnover.copy()
CRSP_noa = impute_missing_var(CRSP_noa, method='zero', compute_variables=['ivaoq'])

operating_assets = CRSP_noa['atq']-CRSP_noa['cheq']-CRSP_noa['ivaoq']
operating_liabs = CRSP_noa['atq']-CRSP_noa['dlcq']-CRSP_noa['dlttq']-CRSP_noa['mibq']-CRSP_noa['pstkq']-CRSP_noa['ceqq']
CRSP_noa['NOA'] = (operating_assets - operating_liabs) / CRSP_noa['lag_atq']
CRSP_noa['NOA'] = CRSP_noa['NOA'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_lturnover


In [57]:
# NOP
CRSP_nop = CRSP_noa.copy()
CRSP_nop['NOP'] = (CRSP_nop['dvc']+CRSP_nop['prstkc']-CRSP_nop['sstk']) / (CRSP_nop['lag_shrout']*CRSP_nop['lag_mthprc'])
CRSP_nop['NOP'] = CRSP_nop['NOP'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_noa


In [58]:
# O2P
CRSP_o2p = CRSP_nop.copy()
CRSP_o2p.sort_values(['permno','yyyymm'],inplace=True)

CRSP_o2p['delta_pstkrv'] = CRSP_o2p.groupby('permno')[['pstkrv']].diff()
CRSP_o2p['delta_pstkrv'] = CRSP_o2p['delta_pstkrv'].replace(0.0, np.nan)
CRSP_o2p['delta_pstkrv'] = CRSP_o2p.groupby('permno')[['delta_pstkrv']].ffill()
CRSP_o2p['delta_pstkrv'] = CRSP_o2p['delta_pstkrv'].fillna(0.0)

CRSP_o2p['O2P'] = (CRSP_o2p['dvc']+CRSP_o2p['prstkc']-CRSP_o2p['delta_pstkrv']) / (CRSP_o2p['lag_shrout']*CRSP_o2p['lag_mthprc'])
CRSP_o2p['O2P'] = CRSP_o2p['O2P'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_nop


In [59]:
# OA
CRSP_oa = CRSP_o2p.copy()

CRSP_oa['non_cash_CA'] = CRSP_oa['actq']-CRSP_oa['cheq']
CRSP_oa['non_cash_WC'] = CRSP_oa['non_cash_CA']-(CRSP_oa['lctq']+CRSP_oa['dlcq']+CRSP_oa['txpq'])

CRSP_oa.sort_values(['permno','yyyymm'],inplace=True)

CRSP_oa['non_cash_WC_delta'] = CRSP_oa.groupby('permno')[['non_cash_WC']].diff()
CRSP_oa['non_cash_WC_delta'] = CRSP_oa['non_cash_WC_delta'].replace(0.0, np.nan)
CRSP_oa['non_cash_WC_delta'] = CRSP_oa.groupby('permno')[['non_cash_WC_delta']].ffill()
CRSP_oa['non_cash_WC_delta'] = CRSP_oa['non_cash_WC_delta'].fillna(0.0)

CRSP_oa['OA'] = (CRSP_oa['non_cash_WC_delta']-CRSP_oa['dpq']) / CRSP_oa['lag_atq']
CRSP_oa['OA'] = CRSP_oa['OA'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_o2p


In [60]:
# OL
CRSP_ol = CRSP_oa.copy()
CRSP_ol['OL'] = (CRSP_ol['cogsq']+CRSP_ol['xsgaq']) / CRSP_ol['atq']
CRSP_E2P['E2P'] = CRSP_E2P['E2P'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_E2P
del CRSP_oa


In [61]:
# PCM
CRSP_pcm = CRSP_ol.copy()
CRSP_pcm['PCM'] = (CRSP_pcm['saleq']-CRSP_pcm['cogsq']) / CRSP_pcm['saleq']
CRSP_pcm['PCM'] = CRSP_pcm['PCM'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_ol


In [62]:
# PM
CRSP_pm = CRSP_pcm.copy()
CRSP_pm['PM'] = CRSP_pm['oiadpq'] / CRSP_pm['saleq']
CRSP_pm['PM'] = CRSP_pm['PM'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_pcm


In [63]:
# PM_adj
CRSP_pm_adj = CRSP_pm.copy()
CRSP_pm_adj.sort_values(['yyyymm','sic2'],inplace=True)
pm_mean = CRSP_pm_adj.groupby(['yyyymm','sic2'],as_index=False)['PM'].mean()
pm_adj_merged = pd.merge(CRSP_pm_adj, pm_mean, on=['yyyymm','sic2'],how='left',suffixes=('', '_mean'))

pm_adj_merged.sort_values(['permno','yyyymm'],inplace=True)
pm_adj_merged['PM_adj'] = pm_adj_merged['PM'] - pm_adj_merged['PM_mean']
pm_adj_merged['PM_adj'] = pm_adj_merged['PM_adj'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_pm


In [64]:
# Prof
CRSP_prof = pm_adj_merged.copy()
CRSP_prof.sort_values(['permno','yyyymm'],inplace=True)

CRSP_prof['Prof'] = CRSP_prof['gp'] / CRSP_prof['BV']
CRSP_prof['Prof'] = CRSP_prof['Prof'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del pm_adj_merged


In [65]:
# Q
CRSP_tobin_q = CRSP_prof.copy()
CRSP_tobin_q.sort_values(['permno','yyyymm'],inplace=True)

CRSP_tobin_q['Q'] = (CRSP_tobin_q['atq']+CRSP_tobin_q['lag_shrout']*CRSP_tobin_q['lag_mthprc']-CRSP_tobin_q['ceqq']-CRSP_tobin_q['txdbq']) / CRSP_tobin_q['atq']
CRSP_tobin_q['Q'] = CRSP_tobin_q['Q'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_prof


In [66]:
# Rel to High
CRSP_rel2high = CRSP_tobin_q.copy()
CRSP_rel2high.sort_values(['permno','yyyymm'],inplace=True)

CRSP_rel2high['52week_high'] = CRSP_rel2high.groupby('permno')[['lag_mthprc']].rolling(window=12).max().reset_index(level=0, drop=True)
CRSP_rel2high['Rel to High'] = CRSP_rel2high['lag_mthprc'] / CRSP_rel2high['52week_high']
CRSP_rel2high['Rel to High'] = CRSP_rel2high['Rel to High'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_tobin_q


In [67]:
CRSP_rel2high.to_csv('rel2high_finished.csv')

In [68]:
# Ret_max
ret_max = CRSP_daily_with_rf.copy().reset_index()
ret_max['yyyymm'] = (ret_max['yyyymmdd'] // 100).astype('int64')
ret_max.sort_values(['permno','yyyymm'],inplace=True)

ret_max_grouped = ret_max.groupby(['permno','yyyymm'])[['dlyret']].max().reset_index()
ret_max_grouped.sort_values(['yyyymm','permno'],inplace=True)
CRSP_rel2high.sort_values(['yyyymm_l_mon','permno'],inplace=True)
ret_max_merged = pd.merge_asof(CRSP_rel2high, ret_max_grouped[['permno','yyyymm','dlyret']],by='permno',
                               left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_retmax'),direction='backward')
ret_max_merged['Ret_max'] = ret_max_merged['dlyret']
ret_max_merged['Ret_max'] = ret_max_merged['Ret_max'].replace([np.inf, -np.inf], np.nan)

In [69]:
ret_max_merged.to_csv('ret_max_finished.csv')

In [70]:
# RNA
CRSP_rna = ret_max_merged.copy()
rna_vars = ['oiadpq','atq','lag_atq','cheq','ivaoq','dlcq','dlttq','mibq','pstkq','ceqq']
CRSP_rna = create_one_more_lag(CRSP_rna, rna_vars)

# lagged_operating_assets = lagged_AT - lagged_CHE - lagged_IVAO
lag_operating_assets = CRSP_rna['lag_atq']-CRSP_rna['lag_cheq']-CRSP_rna['lag_ivaoq']
# lagged_operating_liabilities = lagged_AT - lagged_DLC - lagged_DLTT - lagged_MIB - lagged_PSTK - lagged_CEQ
lag_operating_liabs = CRSP_rna['lag_atq']-CRSP_rna['lag_dlcq']-CRSP_rna['lag_dlttq']-CRSP_rna['lag_mibq']-CRSP_rna['lag_pstkq']-CRSP_rna['lag_ceqq']

CRSP_rna['RNA'] = CRSP_rna['oiadpq'] / (lag_operating_assets-lag_operating_liabs)
CRSP_rna['RNA'] = CRSP_rna['RNA'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del ret_max_merged


In [71]:
# ROA
CRSP_roa = CRSP_rna.copy()
CRSP_roa['ROA'] = CRSP_roa['ibq'] / CRSP_roa['lag_atq']
CRSP_roa['ROA'] = CRSP_roa['ROA'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_rna


In [72]:
# ROC
CRSP_roc = CRSP_roa.copy()
CRSP_roc['ROC'] = (CRSP_roc['mkvaltq']+CRSP_roc['dlttq']-CRSP_roc['atq']) / CRSP_roc['cheq']
CRSP_roc['ROC'] = CRSP_roc['ROC'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_roa


In [73]:
# ROE
CRSP_roe = CRSP_roc.copy()
roe_vars = ['ibq','BV']
CRSP_roe = create_one_more_lag(CRSP_roe,roe_vars)

CRSP_roe['ROE'] = CRSP_roe['ibq'] / CRSP_roe['lag_BV']
CRSP_roe['ROE'] = CRSP_roe['ROE'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_roc


In [74]:
# ROIC
CRSP_roic = CRSP_roe.copy()
CRSP_roic['ROIC'] = (CRSP_roic['ebit']-CRSP_roic['nopiq']) / (CRSP_roic['ceqq']+CRSP_roic['ltq']+CRSP_roic['cheq'])
CRSP_roic['ROIC'] = CRSP_roic['ROIC'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_roe


In [75]:
def rolling_cum_return(df, groupcol, col, window):
    """Vectorized rolling (1+r)-product minus 1, grouped by `groupcol`.

    Replaces `df.groupby(groupcol)[[col]].apply(lambda x: (x+1).rolling(window)
    .apply(lambda y: y.prod()) - 1)`, a doubly Python-level operation (an outer
    groupby().apply() wrapping an inner rolling().apply()) that does not vectorize
    across groups or window positions. Uses the identity
        prod(1+r_i) = exp(sum(log(1+r_i)))
    so the rolling product becomes a rolling SUM of logs, which pandas computes
    natively via groupby(...).rolling(...).sum() (vectorized, no Python callback).
    Safe here because monthly returns in this data are always > -1 (checked:
    crsp_monthly.parquet's ret_adj has min ~-0.9999, never <= -1), so log1p is always
    defined. Validated against the original nested-apply logic on synthetic
    multi-permno data before use: matches to ~1e-15 (floating-point precision).
    """
    logr = np.log1p(df[col])
    tmp = df[[groupcol]].copy()
    tmp['_logr'] = logr
    rollsum = tmp.groupby(groupcol)['_logr'].rolling(window, min_periods=window).sum().reset_index(level=0, drop=True)
    return np.expm1(rollsum)


In [76]:
# r12_2
CRSP_r_12_2 = CRSP_roic.copy()
CRSP_r_12_2.sort_values(['permno','yyyymm'],inplace=True)
# VECTORIZED (was: groupby().apply() wrapping rolling().apply(prod) -- see
# rolling_cum_return's docstring above for the log-sum-exp approach and validation).
CRSP_r_12_2['r_12'] = rolling_cum_return(CRSP_r_12_2, 'permno', 'lag_mthret', 12)
CRSP_r_12_2['r_2'] = rolling_cum_return(CRSP_r_12_2, 'permno', 'lag_mthret', 2)
CRSP_r_12_2['r_12_2'] = CRSP_r_12_2['r_12'] / CRSP_r_12_2['r_2']
CRSP_r_12_2['r_12_2'] = CRSP_r_12_2['r_12_2'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_roic


In [77]:
# r12_7
CRSP_r_12_7 = CRSP_r_12_2.copy()
CRSP_r_12_7.sort_values(['permno','yyyymm'],inplace=True)
# VECTORIZED -- see rolling_cum_return's docstring above.
CRSP_r_12_7['r_7'] = rolling_cum_return(CRSP_r_12_7, 'permno', 'lag_mthret', 7)
CRSP_r_12_7['r_12_7'] = CRSP_r_12_7['r_12'] / CRSP_r_12_7['r_7']
CRSP_r_12_7['r_12_7'] = CRSP_r_12_7['r_12_7'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_r_12_2


In [78]:
# r6_2
CRSP_r_6_2 = CRSP_r_12_7.copy()
CRSP_r_6_2.sort_values(['permno','yyyymm'],inplace=True)
# VECTORIZED -- see rolling_cum_return's docstring above.
CRSP_r_6_2['r_6'] = rolling_cum_return(CRSP_r_6_2, 'permno', 'lag_mthret', 6)
CRSP_r_6_2['r_6_2'] = CRSP_r_6_2['r_6'] / CRSP_r_6_2['r_2']
CRSP_r_6_2['r_6_2'] = CRSP_r_6_2['r_6_2'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_r_12_7


In [79]:
# r2_1
CRSP_r_2_1 = CRSP_r_6_2.copy()
CRSP_r_2_1.sort_values(['permno','yyyymm'],inplace=True)

CRSP_r_2_1['r_2_1'] = CRSP_r_2_1['lag_mthret']
CRSP_r_2_1['r_2_1'] = CRSP_r_2_1['r_2_1'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_r_6_2


In [80]:
# r36_13
CRSP_r_36_13 = CRSP_r_2_1.copy()
CRSP_r_36_13.sort_values(['permno','yyyymm'],inplace=True)
# VECTORIZED -- see rolling_cum_return's docstring above.
CRSP_r_36_13['r_36'] = rolling_cum_return(CRSP_r_36_13, 'permno', 'lag_mthret', 36)
CRSP_r_36_13['r_13'] = rolling_cum_return(CRSP_r_36_13, 'permno', 'lag_mthret', 13)
CRSP_r_36_13['r_36_13'] = CRSP_r_36_13['r_36'] / CRSP_r_36_13['r_13']
CRSP_r_36_13['r_36_13'] = CRSP_r_36_13['r_36_13'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_r_2_1


In [81]:
# S2C
CRSP_s2c = CRSP_r_36_13.copy()
CRSP_s2c.sort_values(['permno','yyyymm'],inplace=True)

CRSP_s2c['S2C'] = CRSP_s2c['saleq'] / CRSP_s2c['cheq']
CRSP_s2c['S2C'] = CRSP_s2c['S2C'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_r_36_13


In [82]:
# S2P
CRSP_s2p = CRSP_s2c.copy()
CRSP_s2p.sort_values(['permno','yyyymm'],inplace=True)

CRSP_s2p['S2P'] = CRSP_s2p['saleq'] / (CRSP_s2p['lag_shrout']*CRSP_s2p['lag_mthprc'])
CRSP_s2p['S2P'] = CRSP_s2p['S2P'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_s2c


In [83]:
# Sales_g
CRSP_sales_g = CRSP_s2p.copy()
CRSP_sales_g.sort_values(['permno','yyyymm'],inplace=True)

CRSP_sales_g['Sales_g'] = CRSP_sales_g.groupby('permno')[['sale']].pct_change()
CRSP_sales_g['Sales_g'] = CRSP_sales_g['Sales_g'].replace(0.0, np.nan)
CRSP_sales_g['Sales_g'] = CRSP_sales_g.groupby('permno')[['Sales_g']].ffill()
CRSP_sales_g['Sales_g'] = CRSP_sales_g['Sales_g'].fillna(0.0)
CRSP_sales_g['Sales_g'] = CRSP_sales_g['Sales_g'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_s2p


In [84]:
# SAT
CRSP_sat= CRSP_sales_g.copy()
CRSP_sat.sort_values(['permno','yyyymm'],inplace=True)

CRSP_sat['SAT'] = CRSP_sat['saleq'] / CRSP_sat['atq']
CRSP_sat['SAT'] = CRSP_sat['SAT'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_sales_g


In [85]:
# SAT_adj
CRSP_sat_adj = CRSP_sat.copy()

CRSP_sat_adj.sort_values(['yyyymm','sic2'],inplace=True)
sat_mean = CRSP_sat_adj.groupby(['yyyymm','sic2'],as_index=False)['SAT'].mean()
sat_adj_merged = pd.merge(CRSP_sat_adj, sat_mean, on=['yyyymm','sic2'],how='left',suffixes=('', '_mean'))

sat_adj_merged['SAT_adj'] = sat_adj_merged['SAT'] - sat_adj_merged['SAT_mean']
sat_adj_merged.sort_values(['permno','yyyymm'],inplace=True)
sat_adj_merged['SAT_adj'] = sat_adj_merged['SAT_adj'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del CRSP_sat


In [86]:
# SGA2S
CRSP_sga2s= sat_adj_merged.copy()
CRSP_sga2s.sort_values(['permno','yyyymm'],inplace=True)

CRSP_sga2s['SGA2S'] = CRSP_sga2s['xsgaq'] / CRSP_sga2s['saleq']
CRSP_sga2s['SGA2S'] = CRSP_sga2s['SGA2S'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del sat_adj_merged


In [87]:
CRSP_sga2s.to_csv('sga2s_finished.csv')

In [88]:
# Spread
# crsp_daily.parquet has neither dlybid/dlyask nor daily high/low prices, so no bid-ask
# spread proxy can be computed from this data source at all (see the note above this cell).
# Left as NaN; resolves to 0 via the same impute_missing_var/final-0-fill pipeline used for
# every other missing fundamental in this notebook -- flagged here as a genuine data gap.
spread_mean_merged = CRSP_sga2s.copy()
spread_mean_merged['Spread'] = np.nan

In [89]:
# Std_turnover
std_turnover = CRSP_daily_with_rf.copy().reset_index()
std_turnover.sort_values(['permno','yyyymmdd'],inplace=True)
std_turnover['dly_turnover'] = std_turnover['dlyvol'] / std_turnover['shrout']

# Vectorized replacement for the previous per-window Python-loop computation.
# Equivalence verified against the original loop logic on synthetic data before
# applying here (same statistic: 21-day rolling std of dly_turnover, with groups
# shorter than 21 rows falling back to the raw value, exactly as before).
std_turnover['dlyturnover_mean'] = std_turnover.groupby('permno')['dly_turnover'].rolling(21).mean().reset_index(level=0, drop=True)
_sizes = std_turnover.groupby('permno')['dly_turnover'].transform('size')
_rolling_std = std_turnover.groupby('permno')['dly_turnover'].rolling(21).std().reset_index(level=0, drop=True)
std_turnover['Std_turnover'] = _rolling_std.where(_sizes >= 21, std_turnover['dly_turnover'])
std_turnover['Std_turnover'] = std_turnover['Std_turnover'].replace([np.inf, -np.inf], np.nan)
del _sizes, _rolling_std


In [90]:
# Std_turnover (continued)
std_turnover['yyyymmdd'] =  pd.to_datetime(std_turnover['yyyymmdd'].astype(str), format='%Y%m%d')
std_turnover.sort_values(['permno','yyyymmdd'],inplace=True)
std_turnover.set_index(['yyyymmdd'],inplace=True)
std_turnover_mon = std_turnover.groupby('permno').resample('ME').last().reset_index()
std_turnover_mon['yyyymm'] = (std_turnover_mon['yyyymmdd'].dt.strftime('%Y%m%d').astype(int)) // 100

std_turnover_mon.sort_values(['yyyymm','permno'],inplace=True)
spread_mean_merged.sort_values(['yyyymm_l_mon','permno'],inplace=True)
std_turnover_merged = pd.merge_asof(spread_mean_merged, std_turnover_mon[['permno','yyyymm','Std_turnover']], by='permno',
                               left_on='yyyymm_l_mon', right_on='yyyymm', suffixes=('','_std_turnover'), direction='backward')

In [91]:
# Std_volume
std_volume = CRSP_daily_with_rf.copy().reset_index()
std_volume.sort_values(['permno','yyyymmdd'],inplace=True)

# Vectorized replacement for the previous per-window Python-loop computation.
# Equivalence verified against the original loop logic on synthetic data before
# applying here (same statistic: 21-day rolling std of dlyvol, with groups
# shorter than 21 rows falling back to the raw value, exactly as before).
std_volume['dlyvol_mean'] = std_volume.groupby('permno')['dlyvol'].rolling(21).mean().reset_index(level=0, drop=True)
_sizes = std_volume.groupby('permno')['dlyvol'].transform('size')
_rolling_std = std_volume.groupby('permno')['dlyvol'].rolling(21).std().reset_index(level=0, drop=True)
std_volume['Std_volume'] = _rolling_std.where(_sizes >= 21, std_volume['dlyvol'])
std_volume['Std_volume'] = std_volume['Std_volume'].replace([np.inf, -np.inf], np.nan)
del _sizes, _rolling_std


In [92]:
# Std_volume (continued)
std_volume['yyyymmdd'] =  pd.to_datetime(std_volume['yyyymmdd'].astype(str), format='%Y%m%d')
std_volume.sort_values(['permno','yyyymmdd'],inplace=True)
std_volume.set_index(['yyyymmdd'],inplace=True)
std_volume_mon = std_volume.groupby('permno').resample('ME').last().reset_index()
std_volume_mon['yyyymm'] = (std_volume_mon['yyyymmdd'].dt.strftime('%Y%m%d').astype(int)) // 100

std_volume_mon.sort_values(['yyyymm','permno'],inplace=True)
std_turnover_merged.sort_values(['yyyymm_l_mon','permno'],inplace=True)
std_volume_merged = pd.merge_asof(std_turnover_merged, std_volume_mon[['permno','yyyymm','Std_volume']], by='permno',
                               left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_std_vol'), direction='backward')

In [93]:
# SUV
# Define a function to perform the rolling regression
# VECTORIZED (was: a per-permno RollingOLS(...).fit() inside suv_rolling_regression,
# called via batched_group_apply below -- see rolling_ols_multi's docstring above).
# suv_rolling_regression itself is no longer used; kept the batched approach out of
# the pipeline entirely since rolling_ols_multi vectorizes across all permnos at once.

# NOTE: the residual-std step below (previously `rolling_resid_std`, a per-window
# Python loop) is now computed directly in the next cell via a vectorized
# Var(X - a - bY) = Var(X) - 2b*Cov(X,Y) + b^2*Var(Y) decomposition, since a is a
# constant offset that drops out of any variance/std calculation. Verified
# equivalent to the original loop logic on synthetic data (incl. multi-permno)
# before applying here, matching to floating-point precision.

suv = CRSP_daily_with_rf.copy()
suv['abs_ret'] = suv['dlyret'].abs()
suv.sort_values(['permno','yyyymmdd'],inplace=True)
suv.reset_index(drop=True,inplace=True)

# rolling_ols_multi (defined above) computes this across all ~10,580 permnos in one
# vectorized pass instead of the previous per-permno RollingOLS/batched_group_apply
# combination -- no separate batching needed since there's no per-group Python object
# construction driving memory use here any more.
suv.reset_index(drop=True,inplace=True)
_suv_coefs = rolling_ols_multi(suv, 'permno', window=21, min_nobs=15, ycol='dlyvol', xcols=['abs_ret'])
reg_params = pd.DataFrame({'const': _suv_coefs['const'], 'abs_ret': _suv_coefs['abs_ret']})
reg_params.reset_index(drop=True,inplace=True)
del _suv_coefs

# The warnings below is due to zero or close-to-zero SSR. It's a fitting issue rather than a calculation/data issue.
# Therefore, we can ignore the warning for now.

In [94]:
# SUV (continued)
suv['suv_const']= reg_params['const']
suv['reg_abs_ret'] = reg_params['abs_ret']
suv['resid'] = suv['dlyvol'] - suv['suv_const'] - suv['reg_abs_ret'] * suv['abs_ret']

# Vectorized rolling residual std (see note in the previous cell): for a fixed
# per-row coefficient b = reg_abs_ret, Std(dlyvol - const - b*abs_ret) over a
# window equals sqrt(Var(dlyvol) - 2*b*Cov(dlyvol,abs_ret) + b^2*Var(abs_ret)),
# since the constant term drops out of any variance. Covariance is computed via
# the moment identity (avoids pandas' grouped-rolling .cov() cross-group
# alignment pitfall) with the standard population->sample (ddof=1) correction.
suv['_xy'] = suv['dlyvol'] * suv['abs_ret']
_g = suv.groupby('permno')
_n      = _g['dlyvol'].rolling(21, min_periods=1).count().reset_index(level=0, drop=True)
_meanX  = _g['dlyvol'].rolling(21, min_periods=1).mean().reset_index(level=0, drop=True)
_meanY  = _g['abs_ret'].rolling(21, min_periods=1).mean().reset_index(level=0, drop=True)
_meanXY = _g['_xy'].rolling(21, min_periods=1).mean().reset_index(level=0, drop=True)
_varX   = _g['dlyvol'].rolling(21, min_periods=1).var().reset_index(level=0, drop=True)
_varY   = _g['abs_ret'].rolling(21, min_periods=1).var().reset_index(level=0, drop=True)
_covXY  = (_n/(_n-1)) * (_meanXY - _meanX*_meanY)

_b = suv['reg_abs_ret']
_var_resid = (_varX - 2*_b*_covXY + (_b**2)*_varY).clip(lower=0)
_rolling_std = np.sqrt(_var_resid)
_sizes = _g['dlyvol'].transform('size')
suv['std'] = _rolling_std.where(_sizes >= 21, suv['dlyvol'])
del suv['_xy'], _g, _n, _meanX, _meanY, _meanXY, _varX, _varY, _covXY, _b, _var_resid, _rolling_std, _sizes

suv['SUV'] = suv['resid'] / suv['std']
suv['SUV'] = suv['SUV'].replace([np.inf, -np.inf], np.nan)

# --- checkpoint (added): if cell 98's resample/merge crashes, reload this
# instead of re-running the SUV regression (cells 96-97) from scratch ---
suv.to_csv('suv_with_signal_checkpoint.csv', index=False)


In [95]:
# SUV (continued)
suv['yyyymmdd'] =  pd.to_datetime(suv['yyyymmdd'].astype(str), format='%Y%m%d')
suv.sort_values(['permno','yyyymmdd'],inplace=True)
suv.set_index(['yyyymmdd'],inplace=True)
suv_monthly = suv.groupby('permno').resample('ME').last().reset_index()
suv_monthly['yyyymmdd'] = suv_monthly['yyyymmdd'].dt.strftime('%Y%m%d').astype(int)
suv_monthly['yyyymm'] = (suv_monthly['yyyymmdd'] // 100).astype('int64')

suv_monthly.sort_values(['yyyymm','permno'],inplace=True)
std_volume_merged.sort_values(['yyyymm_l_mon','permno'],inplace=True)
suv_merged = pd.merge_asof(std_volume_merged,suv_monthly[['permno','yyyymm','SUV']], by='permno',
                           left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_suv'),direction='backward')

In [96]:
# Tan
CRSP_tan = suv_merged.copy()

CRSP_tan['Tan'] = (0.715*CRSP_tan['rectq']+0.547*CRSP_tan['invtq']+0.535*CRSP_tan['ppentq']+CRSP_tan['cheq'])/CRSP_tan['atq']
CRSP_tan['Tan'] = CRSP_tan['Tan'].replace([np.inf, -np.inf], np.nan)
# --- memory cleanup (added; frees no-longer-needed large frames) ---
del suv_merged


In [97]:
# Total vol
total_vol = CRSP_daily_with_rf.copy().reset_index()
total_vol.sort_values(['permno','yyyymmdd'],inplace=True)
total_vol['excess_ret'] = total_vol['dlyret'] - total_vol['rf']

# Vectorized replacement for the previous per-window Python-loop computation.
# Same pattern as Std_turnover/Std_volume above -- verified equivalent to the
# original loop logic on synthetic data before applying here.
total_vol['dlyexret_mean'] = total_vol.groupby('permno')['excess_ret'].rolling(21).mean().reset_index(level=0, drop=True)
_sizes = total_vol.groupby('permno')['excess_ret'].transform('size')
_rolling_std = total_vol.groupby('permno')['excess_ret'].rolling(21).std().reset_index(level=0, drop=True)
total_vol['Total_vol'] = _rolling_std.where(_sizes >= 21, total_vol['excess_ret'])
del _sizes, _rolling_std

total_vol['yyyymmdd'] =  pd.to_datetime(total_vol['yyyymmdd'].astype(str), format='%Y%m%d')
total_vol.sort_values(['permno','yyyymmdd'],inplace=True)
total_vol.set_index(['yyyymmdd'],inplace=True)
total_vol_mon = total_vol.groupby('permno').resample('ME').last().reset_index()
total_vol_mon['yyyymm'] = (total_vol_mon['yyyymmdd'].dt.strftime('%Y%m%d').astype(int)) // 100

total_vol_mon.sort_values(['yyyymm','permno'],inplace=True)
CRSP_tan.sort_values(['yyyymm_l_mon','permno'],inplace=True)
total_vol_merged = pd.merge_asof(CRSP_tan, total_vol_mon[['permno','yyyymm','Total_vol']], by='permno',
                               left_on='yyyymm_l_mon',right_on='yyyymm',suffixes=('','_total_vol'),direction='backward')
total_vol_merged['Total_vol'] = total_vol_merged['Total_vol'].replace([np.inf, -np.inf], np.nan)


In [98]:
total_vol_merged.to_csv('total_vol_merged.csv')

In [99]:
# The calculation is finished!

# Calculated features
freyberger_features = ['AT','A2ME','AOA','ATO','BEME','BEME_adj',
            'Beta_FP','Beta_LN','Beta_daily','C','C2D','CTO','Debt2P','delta_ceq','delta_delGm_minus_delSales',
            'delta_So','delta_shrout','delta_PI2A','DTO','E2P','EPS','Free CF','Idio vol',
            'Investment','IPM','IVC','Lev','LDP','LME','LME_adj','LTurnover','NOA','NOP','O2P',
            'OA','OL','PCM','PM','PM_adj','Prof','Q','Rel to High','Ret_max','RNA','ROA','ROC',
            'ROE','ROIC','r_12_2','r_12_7','r_6_2','r_2_1','r_36_13','S2C','S2P','Sales_g','SAT',
            'SAT_adj','SGA2S','Spread','Std_turnover','Std_volume','Tan','Total_vol']

# Suppose df is your data frame that contains the features
# Final dataframe. 'primaryexch'/'sprtrn' dropped (not in this schema, not used downstream
# -- see cell 1's docstring); 'exchcd' carried in their place as the closest available
# equivalent to primaryexch.
final_df = total_vol_merged[['yyyymm','permno','gvkey','exchcd','mthret','rf','sic2']+freyberger_features]
final_df.sort_values(['yyyymm','permno'],inplace=True)
final_df = impute_missing_var(final_df, method='xsection', compute_variables=freyberger_features)
final_df.replace([np.nan], 0, inplace=True)

,yyyymm,permno,gvkey,exchcd,mthret,rf,sic2,AT,A2ME,AOA,...,Sales_g,SAT,SAT_adj,SGA2S,Spread,Std_turnover,Std_volume,Tan,Total_vol,yyyy_lag1
0,200301,10001,012994,3,0.148143,0.0010,49,48.7625,0.000368,0.062078,...,0.000000,0.130977,-0.282273,1.491697,0.0,5.146389,2.697850e+05,0.0,0.036115,2002
1,200301,10002,019049,3,0.063964,0.0010,60,327.8600,0.000657,0.032066,...,0.000000,0.172355,-0.089742,1.259879,0.0,3.060562,1.800727e+05,0.0,0.026232,2002
2,200301,10012,011907,3,-0.094203,0.0010,36,399.9230,0.015244,0.034147,...,0.000000,0.690023,0.457126,0.208512,0.0,4.789535,2.049207e+05,0.0,0.024291,2002
3,200301,10025,011903,3,-0.487376,0.0010,30,386.9330,0.007309,0.033215,...,0.000000,0.566677,0.293737,0.246450,0.0,6.622853,2.468171e+05,0.0,0.036430,2002
4,200301,10026,012825,3,-0.257071,0.0010,20,268.2270,0.001135,0.007177,...,0.000000,0.041890,-0.187673,4.454521,0.0,2.587340,4.268749e+04,0.0,0.025939,2002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1065774,202412,93374,184899,1,-0.102210,0.0037,63,15157.3000,0.002098,0.012436,...,-0.210153,0.106371,-0.019145,0.000000,0.0,2.681829,2.762392e+05,0.0,0.012628,2023
1065775,202412,93397,029962,3,-0.117446,0.0037,99,307.3130,0.000613,0.026618,...,-0.025481,0.205995,0.075480,0.417897,0.0,1.392086,2.510349e+04,0.0,0.026328,2023
1065776,202412,93426,185138,1,0.021768,0.0037,36,462.7890,0.001649,0.033601,...,-0.020773,0.167158,-0.005763,1.380938,0.0,8.052183,9.849679e+04,0.0,0.022908,2023
1065777,202412,93434,184259,3,0.133333,0.0037,99,120.7260,0.007497,0.039503,...,0.030370,0.122757,-0.007758,1.783266,0.0,39.918102,9.117295e+04,0.0,0.167683,2023


In [100]:
ranked_df = final_df.groupby('yyyymm')[freyberger_features].rank(method='min')  # Min rank ensures the smallest value gets rank 1
temp1_df = pd.concat([final_df[['yyyymm','permno','gvkey','exchcd','mthret','rf','sic2']],ranked_df],axis=1)
temp2_df = temp1_df.groupby('yyyymm')[freyberger_features].transform(lambda x: x / (len(x) + 1))
normalized_df = pd.concat([temp1_df[['yyyymm','permno','gvkey','exchcd','mthret','rf','sic2']],temp2_df],axis=1)

normalized_df.to_csv('features.csv')

print('Done.')

Done.
